[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/linear_algebra/04_orthogonality_projections_and_qr/exercises.ipynb)

# Module 04 — Exercises: Orthogonality, Projections, and QR

Forty-three solved problems in four tiers. Every problem carries a statement, a one-line
intuition, a stepwise solution, a boxed answer, a key takeaway, and — wherever the answer is
numeric or algorithmic — a code cell that recomputes it.

Theorem, proof and example numbers refer to
[first_principles.ipynb](first_principles.ipynb). Symbols follow
[the notation register](../../docs/notation.md): norms written $\lVert x \rVert$, transposes
$A^\top$, factorizations $A = QR$, condition number $\kappa_2$.

The preamble below is shared by every code cell in this notebook.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

EPS = np.finfo(float).eps
print(f"machine epsilon = {EPS:.4e}")

machine epsilon = 2.2204e-16


## L0 — Concept Checks

### Problem L0.1 — Two orthogonal vectors and their lengths

**Statement.** Let $u = (1,2)^\top$ and $v = (-2,1)^\top$. Show $u \perp v$ and compute both norms.

**Intuition.** Orthogonality is one dot product; length is the square root of another.

**Solution.**

*Step 1.* $\langle u, v \rangle = (1)(-2) + (2)(1) = 0$, so $u \perp v$ by Definition 3.2.

*Step 2.* $\lVert u \rVert = \sqrt{1 + 4} = \sqrt5$ and $\lVert v \rVert = \sqrt{4 + 1} = \sqrt5$.

$$
\boxed{\langle u, v \rangle = 0, \qquad \lVert u \rVert = \lVert v \rVert = \sqrt5}
$$

**Key takeaway.** In $\mathbb{R}^2$, rotating a vector by ninety degrees sends $(a,b)$ to
$(-b,a)$, which is why these two are automatically perpendicular and of equal length.

In [2]:
u = np.array([1.0, 2.0])
v = np.array([-2.0, 1.0])
print("<u,v>      :", u @ v)
print("||u||,||v||:", np.linalg.norm(u), np.linalg.norm(v), "   sqrt5 =", np.sqrt(5))
assert abs(u @ v) < 1e-15
assert abs(np.linalg.norm(u) - np.sqrt(5)) < 1e-15

<u,v>      : 0.0
||u||,||v||: 2.23606797749979 2.23606797749979    sqrt5 = 2.23606797749979


### Problem L0.2 — Normalizing a vector

**Statement.** Find the unit vector in the direction of $v = (1,-2,2)^\top$ and verify its length.

**Intuition.** Dividing by the length rescales to one without turning the vector.

**Solution.**

*Step 1.* $\lVert v \rVert = \sqrt{1 + 4 + 4} = 3$.

*Step 2.* $u = v / 3 = (\tfrac13, -\tfrac23, \tfrac23)^\top$, and
$\lVert u \rVert^2 = \tfrac19 + \tfrac49 + \tfrac49 = 1$.

$$
\boxed{u = \tfrac13 (1, -2, 2)^\top, \qquad \lVert u \rVert = 1}
$$

**Key takeaway.** Every Gram-Schmidt step ends with this operation; it is the division by
$r_{kk}$ in Proof 5.3.

In [3]:
v = np.array([1.0, -2.0, 2.0])
u = v / np.linalg.norm(v)
print("||v|| :", np.linalg.norm(v))
print("u     :", u, "   ||u|| :", np.linalg.norm(u))
assert abs(np.linalg.norm(v) - 3.0) < 1e-15
assert abs(np.linalg.norm(u) - 1.0) < 1e-15

||v|| : 3.0
u     : [ 0.3333 -0.6667  0.6667]    ||u|| : 1.0


### Problem L0.3 — The angle between two vectors

**Statement.** Compute the angle between $x = (1,1,0)^\top$ and $y = (0,1,1)^\top$.

**Intuition.** Cauchy-Schwarz makes the normalized inner product a cosine.

**Solution.**

*Step 1.* $\langle x, y \rangle = 1$ and $\lVert x \rVert = \lVert y \rVert = \sqrt2$.

*Step 2.* $\cos\theta = 1 / (\sqrt2 \cdot \sqrt2) = \tfrac12$, so $\theta = \pi/3$.

$$
\boxed{\theta = \pi/3 = 60 \text{ degrees}}
$$

**Key takeaway.** Theorem 4.1 is what guarantees the quotient lies in $[-1,1]$, so the arccosine
is defined; this is Example 6.1.

In [4]:
x = np.array([1.0, 1.0, 0.0])
y = np.array([0.0, 1.0, 1.0])
cos_t = (x @ y) / (np.linalg.norm(x) * np.linalg.norm(y))
print("cos theta :", cos_t, "   theta (degrees) :", np.degrees(np.arccos(cos_t)))
assert abs(cos_t - 0.5) < 1e-15
assert abs(np.degrees(np.arccos(cos_t)) - 60.0) < 1e-12

cos theta : 0.4999999999999999    theta (degrees) : 60.00000000000001


### Problem L0.4 — Scalar and vector projection onto a line

**Statement.** For $u = (2,1)^\top$ and $v = (1,3)^\top$, compute the scalar projection
$\operatorname{comp}_v u$ and the vector projection $\operatorname{proj}_v u$.

**Intuition.** The scalar projection is a signed length along $v$; the vector projection is that
length carried by the unit vector $v/\lVert v \rVert$.

**Solution.**

*Step 1.* $\langle u, v \rangle = 2 + 3 = 5$ and $\lVert v \rVert^2 = 10$.

*Step 2.* $\operatorname{comp}_v u = 5/\sqrt{10} = \sqrt{10}/2$.

*Step 3.* $\operatorname{proj}_v u = \tfrac{5}{10} v = (\tfrac12, \tfrac32)^\top$.

$$
\boxed{\operatorname{comp}_v u = \tfrac{\sqrt{10}}{2}, \qquad \operatorname{proj}_v u = (\tfrac12, \tfrac32)^\top}
$$

**Key takeaway.** The vector projection is $P_W u$ for $W = \operatorname{span}(v)$, so by
Theorem 4.5 it is the closest point of the line to $u$.

In [5]:
u = np.array([2.0, 1.0])
v = np.array([1.0, 3.0])
comp = (u @ v) / np.linalg.norm(v)
proj = ((u @ v) / (v @ v)) * v
print("comp :", comp, "   sqrt10/2 =", np.sqrt(10) / 2)
print("proj :", proj, "   residual . v =", (u - proj) @ v)
assert abs(comp - np.sqrt(10) / 2) < 1e-14
assert np.allclose(proj, [0.5, 1.5])
assert abs((u - proj) @ v) < 1e-14

comp : 1.5811388300841895    sqrt10/2 = 1.5811388300841898
proj : [0.5 1.5]    residual . v = 0.0


### Problem L0.5 — Orthogonal matrices preserve length

**Statement.** Let $Q \in \mathbb{R}^{n \times n}$ satisfy $Q^\top Q = I$. Show
$\lVert Qx \rVert = \lVert x \rVert$ for every $x$, and $\det Q = \pm 1$.

**Intuition.** An isometry cannot change lengths, so it cannot change volumes by more than a sign.

**Solution.**

*Step 1.* $\lVert Qx \rVert^2 = x^\top Q^\top Q x = x^\top x = \lVert x \rVert^2$.

*Step 2.* Taking determinants in $Q^\top Q = I$ and using $\det(XY) = \det X \det Y$ together
with $\det(X^\top) = \det X$ gives $(\det Q)^2 = 1$.

$$
\boxed{\lVert Qx \rVert = \lVert x \rVert, \qquad \det Q = \pm 1}
$$

**Key takeaway.** Step 1 is the whole reason orthogonal transformations are numerically safe; it
is Step 3 of Proof 5.10. The determinant facts used in Step 2 are standard (Strang,
*Introduction to Linear Algebra*, 5th ed., section 5.1) and are not needed anywhere else in this
module.

In [6]:
Qr, _ = np.linalg.qr(rng.standard_normal((5, 5)))
xr = rng.standard_normal(5)
print("||Q x|| - ||x|| :", abs(np.linalg.norm(Qr @ xr) - np.linalg.norm(xr)))
print("det Q           :", np.linalg.det(Qr))
assert abs(np.linalg.norm(Qr @ xr) - np.linalg.norm(xr)) < 1e-14
assert abs(abs(np.linalg.det(Qr)) - 1.0) < 1e-12

||Q x|| - ||x|| : 2.220446049250313e-16
det Q           : 1.0000000000000007


### Problem L0.6 — A projection scales only by $0$ or $1$

**Statement.** Let $P^2 = P$ and suppose $Pv = \lambda v$ for a scalar $\lambda$ and some
$v \neq 0$. Show $\lambda \in \{0,1\}$.

**Intuition.** Projecting twice is projecting once, so the scale factor must square to itself.

**Solution.**

*Step 1.* $P^2 v = P(\lambda v) = \lambda^2 v$, while $P^2 = P$ gives $P^2 v = Pv = \lambda v$.

*Step 2.* Hence $(\lambda^2 - \lambda) v = 0$, and $v \neq 0$ forces $\lambda(\lambda - 1) = 0$.

$$
\boxed{\lambda \in \{0, 1\}}
$$

**Key takeaway.** The $1$-directions are the range and the $0$-directions the null space, which is
the content of Step 2 of Proof 5.6.

In [7]:
a_dir = np.array([1.0, 2.0, -1.0])
P = np.outer(a_dir, a_dir) / (a_dir @ a_dir)
print("||P^2 - P||        :", np.linalg.norm(P @ P - P))
print("P a  = 1 * a       :", np.allclose(P @ a_dir, a_dir))
b_perp = np.array([2.0, -1.0, 0.0])
print("a . b_perp         :", a_dir @ b_perp, "   P b_perp = 0 * b_perp :",
      np.allclose(P @ b_perp, 0.0))
assert np.allclose(P @ P, P)
assert np.allclose(P @ a_dir, a_dir)
assert np.allclose(P @ b_perp, np.zeros(3))

||P^2 - P||        : 1.6653345369377348e-16
P a  = 1 * a       : True
a . b_perp         : 0.0    P b_perp = 0 * b_perp : True


### Problem L0.7 — The trace of a projection is its rank

**Statement.** Let $A \in \mathbb{R}^{m \times n}$ have full column rank and
$P = A(A^\top A)^{-1}A^\top$. Show $\operatorname{tr}(P) = n$.

**Intuition.** A projector has $n$ eigen-directions scaled by $1$ and the rest by $0$, and the
trace adds those factors up.

**Solution.**

*Step 1 — trace cyclicity, proved inline.* For conformable $X, Y$,

$$
\operatorname{tr}(XY) = \sum_i \sum_j X_{ij} Y_{ji} = \sum_j \sum_i Y_{ji} X_{ij} = \operatorname{tr}(YX).
$$

*Step 2.* Apply it with $X = A$ and $Y = (A^\top A)^{-1}A^\top$:

$$
\operatorname{tr}(P) = \operatorname{tr}\bigl((A^\top A)^{-1} A^\top A\bigr) = \operatorname{tr}(I_n) = n .
$$

$$
\boxed{\operatorname{tr}(P) = n = \dim \operatorname{Col}(A)}
$$

**Key takeaway.** The trace of an orthogonal projector counts the dimension it projects onto —
the "effective number of parameters" of a linear model.

In [8]:
Ap = rng.standard_normal((7, 3))
Pp = Ap @ np.linalg.inv(Ap.T @ Ap) @ Ap.T
print("trace(P)        :", np.trace(Pp))
print("rank(P)         :", np.linalg.matrix_rank(Pp))
print("trace(I - P)    :", np.trace(np.eye(7) - Pp), "   (should be 7 - 3 = 4)")
assert abs(np.trace(Pp) - 3.0) < 1e-10
assert np.linalg.matrix_rank(Pp) == 3

trace(P)        : 3.0
rank(P)         : 3
trace(I - P)    : 4.0    (should be 7 - 3 = 4)


### Problem L0.8 — The projection matrix onto a line

**Statement.** Build the orthogonal projection matrix onto the line spanned by $a = (3,4)^\top$
and verify $P^2 = P$ and $P^\top = P$.

**Intuition.** For a single direction the formula $A(A^\top A)^{-1}A^\top$ collapses to
$aa^\top / a^\top a$.

**Solution.**

*Step 1.* $aa^\top = \begin{pmatrix} 9 & 12 \\ 12 & 16\end{pmatrix}$ and $a^\top a = 25$.

*Step 2.* $P = \tfrac{1}{25}\begin{pmatrix} 9 & 12 \\ 12 & 16\end{pmatrix}$, which is symmetric by
inspection.

*Step 3.* $P^2 = \tfrac{1}{625}\begin{pmatrix} 225 & 300 \\ 300 & 400\end{pmatrix} = P$.

$$
\boxed{P = \tfrac{1}{25}\begin{pmatrix} 9 & 12 \\ 12 & 16 \end{pmatrix}}
$$

**Key takeaway.** $\operatorname{tr}(P) = (9+16)/25 = 1$, matching Problem L0.7 with $n = 1$.

In [9]:
a_line = np.array([3.0, 4.0])
P_line = np.outer(a_line, a_line) / (a_line @ a_line)
print("P            :\n", P_line)
print("||P^2 - P||  :", np.linalg.norm(P_line @ P_line - P_line))
print("||P^T - P||  :", np.linalg.norm(P_line.T - P_line))
print("trace(P)     :", np.trace(P_line))
assert np.allclose(P_line, np.array([[9.0, 12.0], [12.0, 16.0]]) / 25)
assert np.allclose(P_line @ P_line, P_line)
assert abs(np.trace(P_line) - 1.0) < 1e-15

P            :
 [[0.36 0.48]
 [0.48 0.64]]
||P^2 - P||  : 0.0
||P^T - P||  : 0.0
trace(P)     : 1.0


## L1 — Foundations

### Problem L1.1 — Gram-Schmidt on two vectors in the plane

**Statement.** Apply Gram-Schmidt to $v_1 = (1,1)^\top$ and $v_2 = (1,3)^\top$ to build an
orthonormal basis of $\mathbb{R}^2$.

**Intuition.** Keep the first direction, then strip from the second whatever points along the
first.

**Solution.**

*Step 1.* $r_{11} = \lVert v_1 \rVert = \sqrt2$ and $q_1 = \tfrac{1}{\sqrt2}(1,1)^\top$.

*Step 2.* $r_{12} = q_1^\top v_2 = \tfrac{1+3}{\sqrt2} = 2\sqrt2$, so

$$
u_2 = v_2 - 2\sqrt2 \, q_1 = (1,3)^\top - (2,2)^\top = (-1,1)^\top .
$$

*Step 3.* $r_{22} = \lVert u_2 \rVert = \sqrt2$ and $q_2 = \tfrac{1}{\sqrt2}(-1,1)^\top$.

$$
\boxed{q_1 = \tfrac{1}{\sqrt2}(1,1)^\top, \qquad q_2 = \tfrac{1}{\sqrt2}(-1,1)^\top}
$$

**Key takeaway.** The pivots $r_{11} = r_{22} = \sqrt2$ multiply to $2$, which by Theorem 4.9 is
the area of the parallelogram spanned by $v_1$ and $v_2$.

In [10]:
v1 = np.array([1.0, 1.0])
v2 = np.array([1.0, 3.0])
q1 = v1 / np.linalg.norm(v1)
u2 = v2 - (q1 @ v2) * q1
q2 = u2 / np.linalg.norm(u2)
print("q1, q2      :", q1, q2)
print("q1 . q2     :", q1 @ q2)
print("pivots      :", np.linalg.norm(v1), np.linalg.norm(u2))
print("area = r11*r22 :", np.linalg.norm(v1) * np.linalg.norm(u2),
      "   |det[v1 v2]| :", abs(np.linalg.det(np.column_stack([v1, v2]))))
assert np.allclose(q1, np.array([1.0, 1.0]) / np.sqrt(2))
assert np.allclose(q2, np.array([-1.0, 1.0]) / np.sqrt(2))
assert abs(q1 @ q2) < 1e-15

q1, q2      : [0.7071 0.7071] [-0.7071  0.7071]
q1 . q2     : 4.440892098500626e-16
pivots      : 1.4142135623730951 1.4142135623730951
area = r11*r22 : 2.0000000000000004    |det[v1 v2]| : 2.0


### Problem L1.2 — Projection matrix onto a plane in $\mathbb{R}^3$

**Statement.** Find the orthogonal projection matrix onto
$W = \operatorname{span}\{(1,0,1)^\top, (0,1,1)^\top\}$.

**Intuition.** Stack the spanning vectors as columns and apply Theorem 4.6.

**Solution.**

*Step 1.* With $A = \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1\end{pmatrix}$,

$$
A^\top A = \begin{pmatrix} 2 & 1 \\ 1 & 2 \end{pmatrix},
\qquad
(A^\top A)^{-1} = \tfrac13 \begin{pmatrix} 2 & -1 \\ -1 & 2 \end{pmatrix} .
$$

*Step 2.* $A(A^\top A)^{-1} = \tfrac13 \begin{pmatrix} 2 & -1 \\ -1 & 2 \\ 1 & 1 \end{pmatrix}$.

*Step 3.* Multiplying by $A^\top$,

$$
P = \tfrac13 \begin{pmatrix} 2 & -1 & 1 \\ -1 & 2 & 1 \\ 1 & 1 & 2 \end{pmatrix} .
$$

$$
\boxed{P = \tfrac13 \begin{pmatrix} 2 & -1 & 1 \\ -1 & 2 & 1 \\ 1 & 1 & 2 \end{pmatrix}}
$$

**Key takeaway.** $\operatorname{tr}(P) = 6/3 = 2 = \dim W$, and $I - P$ is the rank-one projector
onto the normal direction $(1,1,-1)^\top$.

In [11]:
A_pl = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
P_pl = A_pl @ np.linalg.inv(A_pl.T @ A_pl) @ A_pl.T
print("P        :\n", 3 * P_pl, "  (times 3)")
print("trace    :", np.trace(P_pl))
nrm = np.array([1.0, 1.0, -1.0])
print("(I-P) acts on the normal:", (np.eye(3) - P_pl) @ nrm, " vs n =", nrm)
assert np.allclose(3 * P_pl, [[2.0, -1.0, 1.0], [-1.0, 2.0, 1.0], [1.0, 1.0, 2.0]])
assert abs(np.trace(P_pl) - 2.0) < 1e-13
assert np.allclose((np.eye(3) - P_pl) @ nrm, nrm)

P        :
 [[ 2. -1.  1.]
 [-1.  2.  1.]
 [ 1.  1.  2.]]   (times 3)
trace    : 2.0
(I-P) acts on the normal: [ 1.  1. -1.]  vs n = [ 1.  1. -1.]


### Problem L1.3 — Thin QR by classical Gram-Schmidt

**Statement.** Compute the thin QR factorization of
$A = \begin{pmatrix} 1 & 1 \\ 1 & 0 \\ 1 & 1 \end{pmatrix}$ by classical Gram-Schmidt.

**Intuition.** The columns of $Q$ are the orthonormalized columns of $A$; $R$ records the
coefficients used along the way.

**Solution.**

*Step 1.* $r_{11} = \lVert a_1 \rVert = \sqrt3$ and $q_1 = \tfrac{1}{\sqrt3}(1,1,1)^\top$.

*Step 2.* $r_{12} = q_1^\top a_2 = \tfrac{2}{\sqrt3}$, so

$$
u_2 = a_2 - \tfrac{2}{3}(1,1,1)^\top = \left(\tfrac13, -\tfrac23, \tfrac13\right)^\top .
$$

*Step 3.* $r_{22} = \lVert u_2 \rVert = \tfrac{\sqrt6}{3}$ and
$q_2 = \tfrac{1}{\sqrt6}(1,-2,1)^\top$.

$$
\boxed{\hat{Q} = \begin{pmatrix} 1/\sqrt3 & 1/\sqrt6 \\ 1/\sqrt3 & -2/\sqrt6 \\ 1/\sqrt3 & 1/\sqrt6 \end{pmatrix}, \quad \hat{R} = \begin{pmatrix} \sqrt3 & 2/\sqrt3 \\ 0 & \sqrt6/3 \end{pmatrix}}
$$

**Key takeaway.** Both pivots are positive, so by Theorem 4.8 this is the unique thin QR of $A$;
any library returning a different one differs only by column signs.

In [12]:
A_gs = np.array([[1.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
q1 = A_gs[:, 0] / np.linalg.norm(A_gs[:, 0])
r12 = q1 @ A_gs[:, 1]
u2 = A_gs[:, 1] - r12 * q1
r22 = np.linalg.norm(u2)
Qg = np.column_stack([q1, u2 / r22])
Rg = np.array([[np.linalg.norm(A_gs[:, 0]), r12], [0.0, r22]])
print("R      :\n", Rg, "   exact:", np.sqrt(3), 2 / np.sqrt(3), np.sqrt(6) / 3)
print("||A - QR||     :", np.linalg.norm(A_gs - Qg @ Rg))
print("||Q^T Q - I||  :", np.linalg.norm(Qg.T @ Qg - np.eye(2)))
assert np.allclose(Rg, [[np.sqrt(3), 2 / np.sqrt(3)], [0.0, np.sqrt(6) / 3]])
assert np.allclose(Qg[:, 1], np.array([1.0, -2.0, 1.0]) / np.sqrt(6))
assert np.linalg.norm(A_gs - Qg @ Rg) < 1e-14

R      :
 [[1.7321 1.1547]
 [0.     0.8165]]    exact: 1.7320508075688772 1.1547005383792517 0.8164965809277259
||A - QR||     : 9.172677046124954e-19
||Q^T Q - I||  : 5.699774299114064e-16


### Problem L1.4 — $I - P$ projects onto the orthogonal complement

**Statement.** Let $P$ be the orthogonal projection onto $W \subseteq \mathbb{R}^n$. Prove that
$I - P$ is the orthogonal projection onto $W^{\perp}$.

**Intuition.** Theorem 4.4 splits $x$ into $Px$ and $x - Px$; the second half is by definition the
$W^{\perp}$ part.

**Solution.**

*Step 1 — symmetry.* $(I-P)^\top = I - P^\top = I - P$.

*Step 2 — idempotence.* $(I-P)^2 = I - 2P + P^2 = I - P$.

*Step 3 — the range.* By Theorem 4.6 the matrix $I-P$ is the orthogonal projector onto
$\operatorname{Col}(I-P)$, so it remains to show that space is $W^{\perp}$. For any $x$ and any
$y \in W$ (so $Py = y$),

$$
y^\top (I-P)x = (Py)^\top (I-P)x = y^\top (P - P^2)x = 0,
$$

hence $\operatorname{Col}(I-P) \subseteq W^{\perp}$. Conversely $z \in W^{\perp}$ has $Pz = 0$ by
uniqueness in Theorem 4.4, so $z = (I-P)z$ lies in the range.

$$
\boxed{I - P = P_{W^{\perp}}, \qquad P + (I-P) = I}
$$

**Key takeaway.** The pair $(P, I-P)$ is Theorem 4.4 written as matrices, which is why
$\lVert x \rVert^2 = \lVert Px \rVert^2 + \lVert (I-P)x \rVert^2$ always holds.

In [13]:
A_c = rng.standard_normal((6, 2))
P_c = A_c @ np.linalg.inv(A_c.T @ A_c) @ A_c.T
M_c = np.eye(6) - P_c
x_c = rng.standard_normal(6)
print("||M^2 - M||           :", np.linalg.norm(M_c @ M_c - M_c))
print("||M^T - M||           :", np.linalg.norm(M_c.T - M_c))
print("||A^T M||             :", np.linalg.norm(A_c.T @ M_c), "  (range of M is orthogonal to W)")
print("Pythagoras defect     :",
      abs(x_c @ x_c - (P_c @ x_c) @ (P_c @ x_c) - (M_c @ x_c) @ (M_c @ x_c)))
assert np.linalg.norm(M_c @ M_c - M_c) < 1e-12
assert np.linalg.norm(A_c.T @ M_c) < 1e-12

||M^2 - M||           : 2.318976402670325e-16
||M^T - M||           : 1.8799552950076986e-16
||A^T M||             : 3.6890899551010847e-16   (range of M is orthogonal to W)
Pythagoras defect     : 8.881784197001252e-16


### Problem L1.5 — Distance from a point to a plane

**Statement.** Find the distance from $b = (1,2,3)^\top$ to the plane
$x_1 - 2x_2 + 2x_3 = 0$ in $\mathbb{R}^3$.

**Intuition.** The plane is the orthogonal complement of its normal, so the distance is the length
of the projection onto the normal.

**Solution.**

*Step 1.* The plane is $W = \{x : n^\top x = 0\} = \operatorname{span}(n)^{\perp}$ with
$n = (1,-2,2)^\top$.

*Step 2.* By Theorem 4.4 the component of $b$ orthogonal to $W$ is its projection onto $n$, of
length $\lvert n^\top b \rvert / \lVert n \rVert$.

*Step 3.* $n^\top b = 1 - 4 + 6 = 3$ and $\lVert n \rVert = 3$, so the distance is $3/3 = 1$.

$$
\boxed{d = 1}
$$

**Key takeaway.** The point-to-hyperplane formula is one projection; the factor $1/\lVert n \rVert$
is just the normalization Problem L0.2 performs.

In [14]:
n_pl = np.array([1.0, -2.0, 2.0])
b_pt = np.array([1.0, 2.0, 3.0])
d = abs(n_pl @ b_pt) / np.linalg.norm(n_pl)
foot = b_pt - (n_pl @ b_pt) / (n_pl @ n_pl) * n_pl
print("distance     :", d)
print("foot on plane:", foot, "   n . foot =", n_pl @ foot)
print("||b - foot|| :", np.linalg.norm(b_pt - foot))
assert abs(d - 1.0) < 1e-15
assert abs(n_pl @ foot) < 1e-14

distance     : 1.0
foot on plane: [0.6667 2.6667 2.3333]    n . foot = 8.881784197001252e-16
||b - foot|| : 0.9999999999999998


### Problem L1.6 — A Householder reflector for an explicit vector

**Statement.** Build the reflector $H = I - 2vv^\top / v^\top v$ that maps $x = (3,4)^\top$ onto
the negative first axis, and verify $Hx = (-5,0)^\top$.

**Intuition.** Reflect across the hyperplane that bisects $x$ and its target.

**Solution.**

*Step 1.* $\lVert x \rVert = 5$ and $x_1 = 3 \gt 0$, so Part A of Proof 5.8 takes $s = +1$ and
$v = x + 5e_1 = (8,4)^\top$, which may be rescaled to $(2,1)^\top$.

*Step 2.* $vv^\top = \begin{pmatrix} 4 & 2 \\ 2 & 1\end{pmatrix}$ and $v^\top v = 5$, so

$$
H = I - \tfrac25 \begin{pmatrix} 4 & 2 \\ 2 & 1 \end{pmatrix}
= \begin{pmatrix} -3/5 & -4/5 \\ -4/5 & 3/5 \end{pmatrix} .
$$

*Step 3.* $Hx = \bigl(-\tfrac95 - \tfrac{16}{5}, \ -\tfrac{12}{5} + \tfrac{12}{5}\bigr)^\top = (-5,0)^\top$.

$$
\boxed{H = \begin{pmatrix} -3/5 & -4/5 \\ -4/5 & 3/5 \end{pmatrix}, \qquad Hx = (-5,0)^\top}
$$

**Key takeaway.** Rescaling $v$ does not change $H$, because $v$ enters only through
$vv^\top / v^\top v$ — which is the rank-one projector of Problem L0.8.

In [15]:
x_h = np.array([3.0, 4.0])
for v_h in (x_h + 5 * np.array([1.0, 0.0]), np.array([2.0, 1.0])):
    H = np.eye(2) - 2 * np.outer(v_h, v_h) / (v_h @ v_h)
    print(f"v = {v_h}   H =\n{H}\n   H x = {H @ x_h}")
    assert np.allclose(H, [[-0.6, -0.8], [-0.8, 0.6]])
    assert np.allclose(H @ x_h, [-5.0, 0.0])
print("H is symmetric and orthogonal:", np.allclose(H.T, H), np.allclose(H.T @ H, np.eye(2)))

v = [8. 4.]   H =
[[-0.6 -0.8]
 [-0.8  0.6]]
   H x = [-5. -0.]
v = [2. 1.]   H =
[[-0.6 -0.8]
 [-0.8  0.6]]
   H x = [-5. -0.]
H is symmetric and orthogonal: True True


### Problem L1.7 — Least squares through QR

**Statement.** Let $A \in \mathbb{R}^{m \times n}$ have full column rank with thin QR
factorization $A = \hat{Q}\hat{R}$. Express the least-squares solution of $Ax \approx b$ using
$\hat{Q}$ and $\hat{R}$ only.

**Intuition.** $\hat{Q}^\top$ rotates the problem into a coordinate system where the answer is a
triangular solve.

**Solution.**

*Step 1.* Substitute $A = \hat{Q}\hat{R}$ into the normal equations of Theorem 4.7:

$$
\hat{R}^\top \hat{Q}^\top \hat{Q} \hat{R} \hat{x} = \hat{R}^\top \hat{Q}^\top b .
$$

*Step 2.* $\hat{Q}^\top\hat{Q} = I_n$ reduces this to
$\hat{R}^\top \hat{R} \hat{x} = \hat{R}^\top \hat{Q}^\top b$.

*Step 3.* Full column rank makes $\hat{R}$ invertible, so cancelling $\hat{R}^\top$ leaves the
triangular system $\hat{R}\hat{x} = \hat{Q}^\top b$, solved by back-substitution.

$$
\boxed{\hat{R}\hat{x} = \hat{Q}^\top b, \qquad \hat{x} = \hat{R}^{-1}\hat{Q}^\top b}
$$

**Key takeaway.** $\kappa_2(\hat{R}) = \kappa_2(A)$ by Theorem 4.10, so this route never squares
the conditioning the way $A^\top A$ does.

In [16]:
A_ls = rng.standard_normal((20, 4))
b_ls = rng.standard_normal(20)
Q_ls, R_ls = np.linalg.qr(A_ls)
x_qr = np.linalg.solve(R_ls, Q_ls.T @ b_ls)
x_ne = np.linalg.solve(A_ls.T @ A_ls, A_ls.T @ b_ls)
x_lib = np.linalg.lstsq(A_ls, b_ls, rcond=None)[0]
print("QR   :", x_qr)
print("NE   :", x_ne)
print("lstsq:", x_lib)
print("max |QR - lstsq| :", np.abs(x_qr - x_lib).max())
print("kappa2(A), kappa2(R):", np.linalg.cond(A_ls), np.linalg.cond(R_ls))
assert np.abs(x_qr - x_lib).max() < 1e-12
assert abs(np.linalg.cond(A_ls) - np.linalg.cond(R_ls)) < 1e-9

QR   : [-0.2793 -0.2652 -0.0463  0.4435]
NE   : [-0.2793 -0.2652 -0.0463  0.4435]
lstsq: [-0.2793 -0.2652 -0.0463  0.4435]
max |QR - lstsq| : 2.7755575615628914e-16
kappa2(A), kappa2(R): 1.5387737371040158 1.5387737371040158


### Problem L1.8 — A Givens rotation that zeroes one entry

**Statement.** Find $c, s$ so that
$G = \begin{pmatrix} c & s \\ -s & c \end{pmatrix}$ sends $(a,b)^\top$ to $(r,0)^\top$, and show
$G^\top G = I$.

**Intuition.** Rotate the plane until the vector lies on the first axis; the radius is unchanged.

**Solution.**

*Step 1.* The second component of $Gx$ is $-a s + b c$, which vanishes when $\tan\theta = b/a$.

*Step 2.* With $r = \sqrt{a^2+b^2}$ take $c = a/r$ and $s = b/r$; then the first component is
$(a^2+b^2)/r = r$.

*Step 3.* $G^\top G$ has diagonal entries $c^2 + s^2 = 1$ and off-diagonal entries
$cs - sc = 0$, so $G^\top G = I$.

$$
\boxed{c = \frac{a}{\sqrt{a^2+b^2}}, \quad s = \frac{b}{\sqrt{a^2+b^2}}, \quad Gx = \bigl(\sqrt{a^2+b^2},\, 0\bigr)^\top}
$$

**Key takeaway.** A Givens rotation touches only two rows, so it is the tool of choice when a
matrix is sparse or when a single entry has to be annihilated — as in Problem L2.6.

In [17]:
for a_g, b_g in [(3.0, 4.0), (0.0, 2.0), (-1.0, 1.0)]:
    r_g = np.hypot(a_g, b_g)
    c_g, s_g = a_g / r_g, b_g / r_g
    G = np.array([[c_g, s_g], [-s_g, c_g]])
    out = G @ np.array([a_g, b_g])
    print(f"(a,b) = ({a_g:5.1f},{b_g:5.1f})  ->  {out}   r = {r_g:.4f}"
          f"   ||G^T G - I|| = {np.linalg.norm(G.T @ G - np.eye(2)):.2e}")
    assert abs(out[1]) < 1e-15
    assert abs(out[0] - r_g) < 1e-14

(a,b) = (  3.0,  4.0)  ->  [ 5. -0.]   r = 5.0000   ||G^T G - I|| = 3.77e-17
(a,b) = (  0.0,  2.0)  ->  [2. 0.]   r = 2.0000   ||G^T G - I|| = 0.00e+00
(a,b) = ( -1.0,  1.0)  ->  [1.4142 0.    ]   r = 1.4142   ||G^T G - I|| = 3.16e-16


### Problem L1.9 — Householder QR of a $2 \times 2$

**Statement.** Compute a QR factorization of $A = \begin{pmatrix} 0 & 3 \\ 4 & 0 \end{pmatrix}$
using a Householder reflector, then normalize it so that $R$ has positive diagonal.

**Intuition.** One reflector clears the single subdiagonal entry; a sign flip afterwards makes the
answer the unique one of Theorem 4.8.

**Solution.**

*Step 1.* $x = a_1 = (0,4)^\top$, $\lVert x \rVert = 4$, and with the convention $s = +1$ when
$x_1 = 0$ we get $v = x + 4e_1 = (4,4)^\top \sim (1,1)^\top$.

*Step 2.* $H_1 = I - \tfrac22 \begin{pmatrix} 1 & 1 \\ 1 & 1\end{pmatrix} = \begin{pmatrix} 0 & -1 \\ -1 & 0\end{pmatrix}$, and

$$
H_1 A = \begin{pmatrix} -4 & 0 \\ 0 & -3 \end{pmatrix} .
$$

*Step 3.* Multiplying on the left by $D = -I$, itself orthogonal, gives
$R = \begin{pmatrix} 4 & 0 \\ 0 & 3\end{pmatrix}$ with positive diagonal, and
$Q = (D H_1)^\top = \begin{pmatrix} 0 & 1 \\ 1 & 0\end{pmatrix}$.

*Step 4.* $QR = \begin{pmatrix} 0 & 3 \\ 4 & 0 \end{pmatrix} = A$.

$$
\boxed{Q = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \qquad R = \begin{pmatrix} 4 & 0 \\ 0 & 3 \end{pmatrix}}
$$

**Key takeaway.** $A$ is already a permutation of a diagonal matrix, so $Q$ is a permutation and
$R$ is diagonal; the sign normalization is exactly what Theorem 4.8 needs to call the answer
unique.

In [18]:
A_hh = np.array([[0.0, 3.0], [4.0, 0.0]])
x_c1 = A_hh[:, 0]
v_c1 = x_c1 + np.linalg.norm(x_c1) * np.array([1.0, 0.0])
H1 = np.eye(2) - 2 * np.outer(v_c1, v_c1) / (v_c1 @ v_c1)
print("H1     :\n", H1)
print("H1 A   :\n", H1 @ A_hh)
D = -np.eye(2)
R_hh = D @ H1 @ A_hh
Q_hh = (D @ H1).T
print("Q      :\n", Q_hh, "\nR      :\n", R_hh)
print("||A - QR||   :", np.linalg.norm(A_hh - Q_hh @ R_hh))
assert np.allclose(R_hh, [[4.0, 0.0], [0.0, 3.0]])
assert np.allclose(Q_hh, [[0.0, 1.0], [1.0, 0.0]])
assert np.linalg.norm(A_hh - Q_hh @ R_hh) < 1e-14

H1     :
 [[ 0. -1.]
 [-1.  0.]]
H1 A   :
 [[-4.  0.]
 [ 0. -3.]]
Q      :
 [[0. 1.]
 [1. 0.]] 
R      :
 [[4. 0.]
 [0. 3.]]
||A - QR||   : 0.0


### Problem L1.10 — Bessel's inequality and the size of its deficit

**Statement.** For an orthonormal set $\{q_1, \dots, q_k\}$ and any $x$, prove
$\sum_{i=1}^{k} \lvert \langle x, q_i \rangle \rvert^2 \le \lVert x \rVert^2$ and identify the
deficit.

**Intuition.** The projection onto a subspace can only be shorter than the vector it came from.

**Solution.**

*Step 1.* Let $p = \sum_{i=1}^{k} \langle x, q_i \rangle q_i$ and $w = x - p$.

*Step 2.* $\langle w, q_j \rangle = \langle x, q_j \rangle - \langle x, q_j \rangle = 0$ for every
$j$, so $w \perp p$.

*Step 3.* Orthonormality collapses the double sum:
$\lVert p \rVert^2 = \sum_{i=1}^{k} \lvert \langle x, q_i \rangle \rvert^2$.

*Step 4.* Pythagoras gives $\lVert x \rVert^2 = \lVert p \rVert^2 + \lVert w \rVert^2$, so the
deficit is exactly $\lVert w \rVert^2 = \operatorname{dist}(x, \operatorname{span}(q_i))^2$.

$$
\boxed{\lVert x \rVert^2 - \sum_{i=1}^{k} \lvert \langle x, q_i \rangle \rvert^2 = \operatorname{dist}\bigl(x, \operatorname{span}(q_1,\dots,q_k)\bigr)^2}
$$

**Key takeaway.** Bessel is Pythagoras with the perpendicular leg thrown away, which is why the
deficit is a squared distance and not merely a non-negative number.

In [19]:
x_b = np.array([1.0, 2.0, 2.0])
qs = [np.ones(3) / np.sqrt(3), np.array([1.0, -1.0, 0.0]) / np.sqrt(2)]
p_b = sum((x_b @ q) * q for q in qs)
w_b = x_b - p_b
lhs = sum((x_b @ q) ** 2 for q in qs)
print("sum of squares :", lhs, "   ||x||^2 :", x_b @ x_b)
print("deficit        :", x_b @ x_b - lhs, "   ||x - p||^2 :", w_b @ w_b, "   exact 1/6 =", 1 / 6)
assert abs((x_b @ x_b - lhs) - w_b @ w_b) < 1e-13
assert abs(w_b @ w_b - 1 / 6) < 1e-13

sum of squares : 8.833333333333336    ||x||^2 : 9.0
deficit        : 0.1666666666666643    ||x - p||^2 : 0.16666666666666666    exact 1/6 = 0.16666666666666666


### Problem L1.11 — The fundamental theorem of linear algebra, part II

**Statement.** For $A \in \mathbb{R}^{m \times n}$ prove
$\operatorname{Null}(A) = \operatorname{Col}(A^\top)^{\perp}$ and
$\operatorname{Null}(A^\top) = \operatorname{Col}(A)^{\perp}$.

**Intuition.** $Ax$ is the list of inner products of $x$ with the rows of $A$, so $Ax = 0$ says
exactly that $x$ is perpendicular to every row.

**Solution.**

*Step 1.* Write $r_1^\top, \dots, r_m^\top$ for the rows of $A$; they span
$\operatorname{Col}(A^\top)$.

*Step 2.* $Ax = 0$ if and only if $r_i^\top x = 0$ for every $i$, which happens if and only if $x$
is orthogonal to every element of their span.

*Step 3.* Hence $\operatorname{Null}(A) = \operatorname{Col}(A^\top)^{\perp}$.

*Step 4.* Applying Step 3 to $A^\top$ gives
$\operatorname{Null}(A^\top) = \operatorname{Col}(A)^{\perp}$.

$$
\boxed{\operatorname{Null}(A) = \operatorname{Col}(A^\top)^{\perp}, \qquad \operatorname{Null}(A^\top) = \operatorname{Col}(A)^{\perp}}
$$

**Key takeaway.** Combined with Theorem 4.4 this splits $\mathbb{R}^n$ into row space plus null
space and $\mathbb{R}^m$ into column space plus left null space — the two orthogonal
decompositions every least-squares argument in this module uses.

In [20]:
A_ft = rng.standard_normal((5, 3))
A_ft[:, 2] = A_ft[:, 0] + 2 * A_ft[:, 1]          # force a null vector
z = np.array([1.0, 2.0, -1.0])
print("A z                :", A_ft @ z)
print("max |row_i . z|    :", np.abs(A_ft @ z).max())
P_row = A_ft.T @ np.linalg.pinv(A_ft.T)
print("||P_row z||        :", np.linalg.norm(P_row @ z), " (z is orthogonal to the row space)")
assert np.abs(A_ft @ z).max() < 1e-13
assert np.linalg.norm(P_row @ z) < 1e-13

A z                : [0. 0. 0. 0. 0.]
max |row_i . z|    : 0.0
||P_row z||        : 2.7336071744532853e-16  (z is orthogonal to the row space)


### Problem L1.12 — Classical and modified Gram-Schmidt agree in exact arithmetic

**Statement.** Show that CGS and MGS produce the same vectors in exact arithmetic, and explain why
MGS is better in floating point.

**Intuition.** MGS subtracts the same total, just in a different order — and the different order
is what keeps the rounding errors from piling up.

**Solution.**

*Step 1 — the two recurrences.* CGS forms
$v_k = a_k - \sum_{j \lt k}(q_j^\top a_k) q_j$ in one sweep. MGS sets $a_k^{(1)} = a_k$ and
iterates $a_k^{(j+1)} = a_k^{(j)} - (q_j^\top a_k^{(j)}) q_j$ for $j = 1, \dots, k-1$.

*Step 2 — the coefficients coincide.* By induction, $a_k^{(j)} = a_k - \sum_{i \lt j}(q_i^\top a_k)q_i$,
and each subtracted term is orthogonal to $q_j$, so $q_j^\top a_k^{(j)} = q_j^\top a_k$.

*Step 3.* Therefore $a_k^{(k)} = a_k - \sum_{j \lt k}(q_j^\top a_k)q_j = v_k$: identical output.

*Step 4 — floating point.* In finite precision the computed $\tilde{q}_j$ are not exactly
orthogonal. CGS measures every coefficient against the *original* $a_k$, so an error made at step
$j$ is never seen again; MGS measures against the *already corrected* $a_k^{(j)}$, so each step
partly repairs the previous ones. Theorem 4.11 records the resulting bounds
$O(u\kappa_2^2)$ against $O(u\kappa_2)$.

$$
\boxed{\text{exact arithmetic: CGS} \equiv \text{MGS}; \qquad \text{floating point: } O(u\kappa_2^2) \text{ against } O(u\kappa_2)}
$$

**Key takeaway.** Two algebraically identical algorithms can differ by seven orders of magnitude in
accuracy at $\kappa_2 = 10^8$; Section 7.3 of the theory notebook measures the gap across the
whole range.

In [21]:
def qr_cgs(A):
    m, n = A.shape
    Q, R = np.zeros((m, n)), np.zeros((n, n))
    for k in range(n):
        v = A[:, k].copy()
        for j in range(k):
            R[j, k] = Q[:, j] @ A[:, k]
            v = v - R[j, k] * Q[:, j]
        R[k, k] = np.linalg.norm(v)
        Q[:, k] = v / R[k, k]
    return Q, R


def qr_mgs(A):
    m, n = A.shape
    V = A.astype(float).copy()
    Q, R = np.zeros((m, n)), np.zeros((n, n))
    for k in range(n):
        R[k, k] = np.linalg.norm(V[:, k])
        Q[:, k] = V[:, k] / R[k, k]
        for j in range(k + 1, n):
            R[k, j] = Q[:, k] @ V[:, j]
            V[:, j] = V[:, j] - R[k, j] * Q[:, k]
    return Q, R


A_well = rng.standard_normal((10, 4))
Qc, Rc = qr_cgs(A_well)
Qm, Rm = qr_mgs(A_well)
print("well conditioned, kappa2 =", f"{np.linalg.cond(A_well):.3e}")
print("  max |R_cgs - R_mgs|      :", np.abs(Rc - Rm).max())
print("  loss CGS, MGS            :", np.linalg.norm(Qc.T @ Qc - np.eye(4)),
      np.linalg.norm(Qm.T @ Qm - np.eye(4)))

U, _ = np.linalg.qr(rng.standard_normal((10, 10)))
V, _ = np.linalg.qr(rng.standard_normal((4, 4)))
A_ill = U[:, :4] @ np.diag(np.logspace(0, -8, 4)) @ V.T
Qc2, _ = qr_cgs(A_ill)
Qm2, _ = qr_mgs(A_ill)
print("ill conditioned, kappa2  =", f"{np.linalg.cond(A_ill):.3e}")
print("  loss CGS                 :", f"{np.linalg.norm(Qc2.T @ Qc2 - np.eye(4)):.3e}")
print("  loss MGS                 :", f"{np.linalg.norm(Qm2.T @ Qm2 - np.eye(4)):.3e}")
assert np.abs(Rc - Rm).max() < 1e-12
assert np.linalg.norm(Qm2.T @ Qm2 - np.eye(4)) < np.linalg.norm(Qc2.T @ Qc2 - np.eye(4))

well conditioned, kappa2 = 2.758e+00
  max |R_cgs - R_mgs|      : 2.220446049250313e-16
  loss CGS, MGS            : 2.8984347435648934e-16 2.6467163779560464e-16
ill conditioned, kappa2  = 1.000e+08
  loss CGS                 : 1.796e-05
  loss MGS                 : 5.489e-10


### Problem L1.13 — Minimality and perpendicularity are the same condition

**Statement.** Let $W \subseteq \mathbb{R}^n$ be a subspace, $v \in \mathbb{R}^n$ and $p \in W$.
Prove that $p$ minimizes $\lVert v - w \rVert$ over $w \in W$ **if and only if** $v - p \perp W$.

**Intuition.** If the error still leans into $W$, you can step along that lean and do better.

**Solution.**

*Step 1 — perpendicular implies minimal.* Assume $v - p \perp W$. For $w \in W$ write
$v - w = (v-p) + (p-w)$ with $p - w \in W$, so the two pieces are orthogonal and

$$
\lVert v - w \rVert^2 = \lVert v - p \rVert^2 + \lVert p - w \rVert^2 \ \ge \ \lVert v - p \rVert^2 ,
$$

with equality only at $w = p$.

*Step 2 — minimal implies perpendicular.* Assume $v - p \not\perp W$ and pick a unit $w_0 \in W$
with $\alpha = \langle v-p, w_0 \rangle \neq 0$. Then $p + \alpha w_0 \in W$ and

$$
\lVert v - p - \alpha w_0 \rVert^2 = \lVert v-p \rVert^2 - \lvert \alpha \rvert^2 \ \lt \ \lVert v-p \rVert^2 ,
$$

so $p$ was not minimal.

$$
\boxed{p = \arg\min_{w \in W} \lVert v - w \rVert \iff v - p \perp W}
$$

**Key takeaway.** This is the sharp two-sided form of Theorem 4.5, and Step 2 is why the normal
equations are *necessary* and not merely sufficient.

In [22]:
W_basis = rng.standard_normal((5, 2))
v_t = rng.standard_normal(5)
P_W = W_basis @ np.linalg.inv(W_basis.T @ W_basis) @ W_basis.T
p_t = P_W @ v_t
print("||W^T (v - p)||   :", np.linalg.norm(W_basis.T @ (v_t - p_t)))
best = np.linalg.norm(v_t - p_t)
worse = [np.linalg.norm(v_t - (p_t + t * W_basis[:, 0])) for t in (-0.3, -0.05, 0.05, 0.3)]
print("distance at p     :", best)
print("distances nearby  :", np.array(worse))
assert np.linalg.norm(W_basis.T @ (v_t - p_t)) < 1e-13
assert all(w > best for w in worse)

||W^T (v - p)||   : 1.602148379482783e-15
distance at p     : 0.9373251553903564
distances nearby  : [1.1977 0.9455 0.9455 1.1977]


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — The normal equations of linear regression, from orthogonality

**Statement.** Derive $X^\top X w = X^\top y$ from the geometric requirement that the residual
$e = y - Xw$ be orthogonal to $\operatorname{Col}(X)$.

**Intuition.** A fitted model has extracted everything the features can explain exactly when what
is left correlates with none of them.

**Solution.**

*Step 1.* $Xw$ ranges over $\operatorname{Col}(X) = \operatorname{span}\{x_1, \dots, x_p\}$, so by
Theorem 4.5 the best $w$ is the one making $e \perp \operatorname{Col}(X)$.

*Step 2.* $e$ is orthogonal to the span exactly when it is orthogonal to each column:
$x_i^\top e = 0$ for $i = 1, \dots, p$.

*Step 3.* Stacking those $p$ scalar equations gives $X^\top e = 0$.

*Step 4.* Substituting $e = y - Xw$ yields $X^\top y - X^\top X w = 0$.

$$
\boxed{X^\top X w = X^\top y}
$$

**Key takeaway.** Every regression diagnostic that plots residuals against a predictor is checking
Step 2 by eye; the plot is flat by construction for predictors already in the model.

In [23]:
n_reg = 120
x1 = rng.standard_normal(n_reg)
x2 = 0.4 * x1 + rng.standard_normal(n_reg)
X_reg = np.column_stack([np.ones(n_reg), x1, x2])
y_reg = 1.0 - 2.0 * x1 + 0.5 * x2 + 0.3 * rng.standard_normal(n_reg)
w_reg = np.linalg.solve(X_reg.T @ X_reg, X_reg.T @ y_reg)
e_reg = y_reg - X_reg @ w_reg
print("fitted w        :", w_reg, "   (true 1, -2, 0.5)")
print("X^T e           :", X_reg.T @ e_reg)
print("corr(x1, e)     :", np.corrcoef(x1, e_reg)[0, 1])
print("corr(x2, e)     :", np.corrcoef(x2, e_reg)[0, 1])
assert np.abs(X_reg.T @ e_reg).max() < 1e-11
assert abs(np.corrcoef(x1, e_reg)[0, 1]) < 1e-12

fitted w        : [ 0.9889 -2.0386  0.5142]    (true 1, -2, 0.5)
X^T e           : [-0.  0. -0.]
corr(x1, e)     : 4.260476904880893e-16
corr(x2, e)     : -2.7714665956807616e-15


### Problem L2.2 — Minimum-norm solution of an underdetermined system

**Statement.** Let $A \in \mathbb{R}^{m \times n}$ have full row rank with $m \lt n$. Find the
solution of $Ax = b$ of smallest Euclidean norm.

**Intuition.** The solution set is a shifted null space; the shortest member is the one with no
null-space component at all.

**Solution.**

*Step 1.* By Problem L1.11 and Theorem 4.4,
$\mathbb{R}^n = \operatorname{Col}(A^\top) \oplus \operatorname{Null}(A)$, so any solution splits
as $x = x_r + x_n$ with $x_r \in \operatorname{Col}(A^\top)$ and $x_n \in \operatorname{Null}(A)$.

*Step 2.* $A x = A x_r$, so $x_r$ is the same for every solution, and Pythagoras gives
$\lVert x \rVert^2 = \lVert x_r \rVert^2 + \lVert x_n \rVert^2$. The minimum is at $x_n = 0$.

*Step 3.* Write $x_{\min} = A^\top z$. Then $AA^\top z = b$, and full row rank makes $AA^\top$
invertible by the argument of Step 3 of Proof 5.6 applied to $A^\top$.

$$
\boxed{x_{\min} = A^\top (AA^\top)^{-1} b}
$$

**Key takeaway.** This is the same projection idea with the roles of the two fundamental subspaces
exchanged; in machine learning it is the implicit bias of gradient descent started at zero on an
overparameterized linear model.

In [24]:
A_un = rng.standard_normal((3, 7))
b_un = rng.standard_normal(3)
x_min = A_un.T @ np.linalg.solve(A_un @ A_un.T, b_un)
print("residual ||A x - b|| :", np.linalg.norm(A_un @ x_min - b_un))
print("||x_min||            :", np.linalg.norm(x_min))
print("||lstsq solution||   :", np.linalg.norm(np.linalg.lstsq(A_un, b_un, rcond=None)[0]))
Qfull, _ = np.linalg.qr(A_un.T, mode="complete")
ns = Qfull[:, 3:]                          # Null(A) = Col(A^T) perp, by Problem L1.11
for _ in range(4):
    z = ns @ rng.standard_normal(4)
    print("   competitor norm   :", np.linalg.norm(x_min + z))
    assert np.linalg.norm(x_min + z) > np.linalg.norm(x_min)
assert np.linalg.norm(A_un @ x_min - b_un) < 1e-12

residual ||A x - b|| : 5.551115123125783e-16
||x_min||            : 1.0773193272227342
||lstsq solution||   : 1.0773193272227348
   competitor norm   : 2.6431583403591254
   competitor norm   : 2.473940211942036
   competitor norm   : 2.0491603613316802
   competitor norm   : 3.1058713850887374


### Problem L2.3 — Best approximation with a prescribed column space

**Statement.** Fix a subspace $S \subseteq \mathbb{R}^m$ with orthogonal projector $P_S$. Show
that among all $B \in \mathbb{R}^{m \times n}$ with $\operatorname{Col}(B) \subseteq S$, the
Frobenius error $\lVert A - B \rVert_F$ is minimized uniquely at $B = P_S A$, with value
$\lVert (I - P_S)A \rVert_F$. Then state precisely what remains to be proved for the rank-$k$
problem.

**Intuition.** The Frobenius norm decouples over columns, so a matrix approximation problem with a
fixed target subspace is $n$ independent copies of Theorem 4.5.

**Solution.**

*Step 1.* $\lVert A - B \rVert_F^2 = \sum_{j=1}^{n} \lVert a_j - b_j \rVert^2$, and the constraint
$\operatorname{Col}(B) \subseteq S$ says each $b_j$ ranges over $S$ independently of the others.

*Step 2.* By Theorem 4.5 each summand is minimized uniquely at $b_j = P_S a_j$, so the joint
minimizer is $B = P_S A$.

*Step 3.* The optimal value is $\sum_j \lVert (I-P_S)a_j \rVert^2 = \lVert (I-P_S)A \rVert_F^2$.

*Step 4 — what is left.* Since every $B$ of rank at most $k$ has $\operatorname{Col}(B)$ inside
some $k$-dimensional $S$, the rank-$k$ problem reduces to

$$
\min_{\dim S = k} \ \lVert (I - P_S) A \rVert_F .
$$

That outer minimization is **not** a projection question and is not solved here: its answer is the
Eckart-Young-Mirsky theorem, which identifies the optimal $S$ as the span of the top $k$ left
singular vectors and is proved in
[Module 07](../07_canonical_forms_and_svd/first_principles.ipynb). The minimizer there is unique
only when $\sigma_k \gt \sigma_{k+1}$.

$$
\boxed{\min_{\operatorname{Col}(B) \subseteq S} \lVert A - B \rVert_F = \lVert (I - P_S)A \rVert_F, \ \text{ attained only at } B = P_S A}
$$

**Key takeaway.** Splitting the low-rank problem this way separates what orthogonality settles
(the best $B$ for a given $S$) from what it does not (the best $S$), and keeps the citation
honest.

In [25]:
A_lr = rng.standard_normal((9, 5))
Sbasis, _ = np.linalg.qr(rng.standard_normal((9, 3)))
P_S = Sbasis @ Sbasis.T
B_star = P_S @ A_lr
print("optimal error   :", np.linalg.norm(A_lr - B_star))
print("||(I-P)A||_F    :", np.linalg.norm((np.eye(9) - P_S) @ A_lr))
for _ in range(4):
    B_other = P_S @ (A_lr + 0.3 * rng.standard_normal((9, 5)))
    print("   competitor   :", np.linalg.norm(A_lr - B_other))
    assert np.linalg.norm(A_lr - B_other) > np.linalg.norm(A_lr - B_star)
assert abs(np.linalg.norm(A_lr - B_star) - np.linalg.norm((np.eye(9) - P_S) @ A_lr)) < 1e-12

optimal error   : 5.740317863011532
||(I-P)A||_F    : 5.740317863011533
   competitor   : 5.8066639864250895
   competitor   : 5.822708368177206
   competitor   : 5.84233203130065
   competitor   : 5.8717077420786365


### Problem L2.4 — Kernel ridge regression and the representer theorem

**Statement.** In a reproducing kernel Hilbert space $\mathcal{H}$ with kernel
$k(x,z) = \langle \Phi(x), \Phi(z) \rangle$, minimize

$$
J(f) = \tfrac12 \sum_{i=1}^{n} \bigl(y_i - f(x_i)\bigr)^2 + \tfrac{\lambda}{2}\lVert f \rVert_{\mathcal{H}}^2
$$

and derive the prediction formula.

**Intuition.** The data can only see the projection of $f$ onto the span of the feature vectors,
and the penalty punishes everything else, so the optimum has nothing else.

**Solution.**

*Step 1 — split $f$.* Let $\mathcal{S} = \operatorname{span}\{\Phi(x_1), \dots, \Phi(x_n)\}$ and
apply Theorem 4.4: $f = f_{\parallel} + f_{\perp}$ with $f_{\parallel} \in \mathcal{S}$ and
$f_{\perp} \in \mathcal{S}^{\perp}$.

*Step 2 — the loss ignores $f_{\perp}$.* The reproducing property gives
$f(x_i) = \langle f, \Phi(x_i) \rangle = \langle f_{\parallel}, \Phi(x_i) \rangle$, so the first
term of $J$ depends on $f_{\parallel}$ only.

*Step 3 — the penalty charges for it.* Pythagoras gives
$\lVert f \rVert^2 = \lVert f_{\parallel} \rVert^2 + \lVert f_{\perp} \rVert^2$, so any $f$ with
$f_{\perp} \neq 0$ is beaten by $f_{\parallel}$. Hence $f^{\star} = \sum_i \alpha_i \Phi(x_i)$.

*Step 4 — solve for $\alpha$.* With $K_{ij} = k(x_i,x_j)$ we get $f(x_i) = (K\alpha)_i$ and
$\lVert f \rVert^2 = \alpha^\top K \alpha$, so

$$
J(\alpha) = \tfrac12 \lVert y - K\alpha \rVert^2 + \tfrac{\lambda}{2}\alpha^\top K \alpha,
\qquad
\nabla_\alpha J = K\bigl((K + \lambda I)\alpha - y\bigr) .
$$

Taking $\alpha = (K+\lambda I)^{-1}y$ makes the gradient vanish, and $K + \lambda I \succ 0$ for
$\lambda \gt 0$.

$$
\boxed{f^{\star}(x_{\ast}) = k(x_{\ast})^\top (K + \lambda I)^{-1} y}
$$

**Key takeaway.** The representer theorem is Theorem 4.4 applied in an infinite-dimensional space:
an infinite-dimensional optimization collapses to an $n \times n$ solve because everything outside
$\mathcal{S}$ is pure cost.

In [26]:
xs = np.linspace(-1.0, 1.0, 12)
ys = np.sin(2.0 * xs) + 0.05 * rng.standard_normal(12)
lam = 0.1


def kern(a, b):
    """Quadratic kernel (1 + a b)^2, whose feature map is (1, sqrt2 a, a^2)."""
    return (1.0 + np.outer(a, b)) ** 2


def feat(a):
    return np.column_stack([np.ones_like(a), np.sqrt(2.0) * a, a ** 2])


K = kern(xs, xs)
alpha = np.linalg.solve(K + lam * np.eye(12), ys)
x_test = np.array([-0.7, 0.0, 0.35, 0.9])
pred_kernel = kern(x_test, xs) @ alpha

Phi = feat(xs)
w_feat = np.linalg.solve(Phi.T @ Phi + lam * np.eye(3), Phi.T @ ys)
pred_feature = feat(x_test) @ w_feat

print("kernel form  :", pred_kernel)
print("feature form :", pred_feature)
print("max gap      :", np.abs(pred_kernel - pred_feature).max())
print("||f||^2 two ways:", alpha @ K @ alpha, w_feat @ w_feat)
assert np.abs(pred_kernel - pred_feature).max() < 1e-10
assert abs(alpha @ K @ alpha - w_feat @ w_feat) < 1e-10

kernel form  : [-0.8444 -0.0184  0.4085  1.098 ]
feature form : [-0.8444 -0.0184  0.4085  1.098 ]
max gap      : 9.992007221626409e-16
||f||^2 two ways: 0.7295520810097109 0.7295520810097074


### Problem L2.5 — The SRHT is a subspace embedding, not a projection

**Statement.** Describe the subsampled randomized Hadamard transform
$S = \sqrt{n/l}\, R H D \in \mathbb{R}^{l \times n}$ with $l \ll n$, state what guarantee it
provides for randomized least squares, and explain why calling it a projection is wrong.

**Intuition.** $HD$ smears every coordinate's energy across all $n$ positions, so uniformly
sampling $l$ of them loses little; the result is a short, nearly length-preserving sketch.

**Solution.**

*Step 1 — the three factors.* $D$ is diagonal with independent random signs; $H$ is the normalized
Walsh-Hadamard matrix, orthogonal with $H^\top H = I_n$; $R$ selects $l$ rows uniformly without
replacement.

*Step 2 — why it is not a projection.* $S$ maps $\mathbb{R}^n \to \mathbb{R}^l$ with $l \lt n$, so
$S^2$ is not even a defined product and $S^2 = S$ cannot be asked. Sampling without replacement
gives $R R^\top = I_l$ and hence

$$
S S^\top = \tfrac{n}{l} R H D D^\top H^\top R^\top = \tfrac{n}{l} I_l ,
$$

so the rows of $S$ are orthogonal but of norm $\sqrt{n/l}$, not $1$. What is true is
$\mathbb{E}[S^\top S] = I_n$.

*Step 3 — the guarantee.* For a fixed $d$-dimensional subspace $\mathcal{V} \subseteq \mathbb{R}^n$
and $l = O(\varepsilon^{-2} d \log d)$, with high probability

$$
(1-\varepsilon)\lVert y \rVert \le \lVert Sy \rVert \le (1+\varepsilon)\lVert y \rVert
\qquad \text{for every } y \in \mathcal{V} .
$$

*Step 4 — the payoff.* Applying this to $\mathcal{V} = \operatorname{Col}([A \ b])$ makes
$\min_x \lVert SAx - Sb \rVert$ a $(1+O(\varepsilon))$-accurate surrogate for
$\min_x \lVert Ax - b \rVert$, at cost $O(n \log n + l d^2)$ instead of $O(nd^2)$.

$$
\boxed{S = \sqrt{\tfrac{n}{l}}\, R H D, \quad S S^\top = \tfrac{n}{l} I_l, \quad \lVert Sy \rVert = (1 \pm \varepsilon)\lVert y \rVert \text{ on } \mathcal{V}}
$$

**Key takeaway.** A sketch preserves lengths approximately on one subspace; a projection preserves
one subspace exactly and destroys its complement. Confusing the two is the misconception the
module README warns about. Sources: Tropp, *Advances in Adaptive Data Analysis* **3** (2011),
115-126; Drineas and Mahoney, *Communications of the ACM* **59**(6) (2016), 80-90.

In [27]:
def hadamard(n):
    """Normalized Walsh-Hadamard matrix for n a power of two."""
    Hm = np.ones((1, 1))
    while Hm.shape[0] < n:
        Hm = np.block([[Hm, Hm], [Hm, -Hm]])
    return Hm / np.sqrt(n)


n_s, d_s = 1024, 10
Hn = hadamard(n_s)
V, _ = np.linalg.qr(rng.standard_normal((n_s, d_s)))     # an orthonormal basis of V
print("||H^T H - I||:", np.linalg.norm(Hn.T @ Hn - np.eye(n_s)))
print(f"{'l':>6s}{'sigma_min':>12s}{'sigma_max':>12s}{'distortion':>13s}{'||S S^T - (n/l) I||':>22s}")
eps_seen = []
for l_s in (32, 64, 128, 256, 512):
    Dg = np.diag(rng.choice([-1.0, 1.0], size=n_s))
    rows = rng.choice(n_s, size=l_s, replace=False)
    Rsel = np.zeros((l_s, n_s))
    Rsel[np.arange(l_s), rows] = 1.0
    S = np.sqrt(n_s / l_s) * Rsel @ Hn @ Dg
    sv = np.linalg.svd(S @ V, compute_uv=False)
    eps = max(sv.max() - 1.0, 1.0 - sv.min())
    eps_seen.append(eps)
    print(f"{l_s:6d}{sv.min():12.4f}{sv.max():12.4f}{eps:13.4f}"
          f"{np.linalg.norm(S @ S.T - (n_s / l_s) * np.eye(l_s)):22.2e}")
print("\nthe distortion shrinks roughly like 1/sqrt(l), as the O(eps^-2 d log d) bound predicts")
assert eps_seen[-1] < eps_seen[0]
assert eps_seen[-1] < 0.2

||H^T H - I||: 0.0


     l   sigma_min   sigma_max   distortion   ||S S^T - (n/l) I||


    32      0.4210      1.4351       0.5790              4.02e-14


    64      0.6204      1.3043       0.3796              0.00e+00


   128      0.7734      1.2215       0.2266              2.05e-14
   256      0.8909      1.1513       0.1513              0.00e+00


   512      0.8829      1.0710       0.1171              1.10e-14

the distortion shrinks roughly like 1/sqrt(l), as the O(eps^-2 d log d) bound predicts


### Problem L2.6 — Updating a QR factorization when a row arrives

**Statement.** Given the thin QR factorization $A = QR$ of $A \in \mathbb{R}^{m \times n}$, design
an algorithm that produces the factorization of $\begin{bmatrix} A \\ a^\top \end{bmatrix}$, and
state its cost.

**Intuition.** Appending a row leaves $R$ triangular except for one extra row at the bottom, and
$n$ Givens rotations sweep that row away one entry at a time.

**Solution.**

*Step 1 — restate the problem.* With $\tilde{Q}_0 = \begin{bmatrix} Q & 0 \\ 0 & 1\end{bmatrix}$
and $\tilde{R}_0 = \begin{bmatrix} R \\ a^\top \end{bmatrix}$ we have
$\begin{bmatrix} A \\ a^\top\end{bmatrix} = \tilde{Q}_0 \tilde{R}_0$, and $\tilde{R}_0$ is upper
triangular apart from its last row.

*Step 2 — the sweep.* For $k = 1, \dots, n$ apply the Givens rotation $G_k$ of Problem L1.8 in the
plane of rows $k$ and $n+1$, chosen to zero the $k$-th entry of the bottom row against the pivot
$R_{kk}$. Rotation $k$ touches only those two rows, so it cannot refill entries $1, \dots, k-1$
that earlier rotations already cleared.

*Step 3 — reassemble.* After $n$ rotations
$G_n \cdots G_1 \tilde{R}_0 = \begin{bmatrix} R_{\text{new}} \\ 0 \end{bmatrix}$, and
$Q_{\text{new}} = \tilde{Q}_0 G_1^\top \cdots G_n^\top$ restricted to its first $n$ columns.

*Step 4 — cost.* Each rotation changes two rows of $\tilde{R}$, costing $O(n)$, and two columns of
$\tilde{Q}$, costing $O(m)$. Over $n$ rotations that is $O(n^2)$ to maintain $R$ alone and
$O(mn)$ more if $Q$ is also carried.

$$
\boxed{O(n^2) \text{ for } R \text{ alone}, \qquad O(n^2 + mn) \text{ when } Q \text{ is maintained}}
$$

**Key takeaway.** Refactoring from scratch costs $O(mn^2)$, so the update is cheaper by a factor
of $m$ — this is what makes recursive least squares and online regression practical. Note that
appending a row is a *row update*, not a rank-one update of $A$; the two are different operations.

In [28]:
def qr_append_row(Q, R, a):
    """Update a thin QR factorization when one row is appended to A."""
    m, n = Q.shape
    Qt = np.zeros((m + 1, n + 1))
    Qt[:m, :n] = Q
    Qt[m, n] = 1.0
    Rt = np.vstack([R, a[None, :]])
    for k in range(n):
        r_kk, r_bk = Rt[k, k], Rt[n, k]
        rad = np.hypot(r_kk, r_bk)
        if rad == 0.0:
            continue
        c, s = r_kk / rad, r_bk / rad
        G = np.array([[c, s], [-s, c]])
        Rt[[k, n], :] = G @ Rt[[k, n], :]
        Qt[:, [k, n]] = Qt[:, [k, n]] @ G.T
    return Qt[:, :n], Rt[:n, :]


A_up = rng.standard_normal((30, 4))
a_new = rng.standard_normal(4)
Q0, R0 = np.linalg.qr(A_up)
Q1, R1 = qr_append_row(Q0, R0, a_new)
A_big = np.vstack([A_up, a_new[None, :]])
Q_ref, R_ref = np.linalg.qr(A_big)

print("||A_new - Q1 R1||     :", np.linalg.norm(A_big - Q1 @ R1))
print("||Q1^T Q1 - I||       :", np.linalg.norm(Q1.T @ Q1 - np.eye(4)))
print("R1 upper triangular?  :", np.allclose(R1, np.triu(R1)))
print("|R1| vs |R_ref| (signs aside):", np.abs(np.abs(R1) - np.abs(R_ref)).max())
assert np.linalg.norm(A_big - Q1 @ R1) < 1e-12
assert np.linalg.norm(Q1.T @ Q1 - np.eye(4)) < 1e-12
assert np.abs(np.abs(R1) - np.abs(R_ref)).max() < 1e-11

||A_new - Q1 R1||     : 3.4132869806949585e-15
||Q1^T Q1 - I||       : 8.870599057179003e-16
R1 upper triangular?  : True
|R1| vs |R_ref| (signs aside): 8.881784197001252e-16


### Problem L2.7 — Loss of orthogonality on the Lauchli matrix

**Statement.** For the Lauchli matrix $A_\varepsilon$ of Proposition 4.12 with
$\varepsilon = 10^{-8}$, compare $\lVert \hat{Q}^\top\hat{Q} - I \rVert_2$ for classical
Gram-Schmidt, modified Gram-Schmidt and Householder, and explain the ordering using
$\kappa_2(A_\varepsilon)$.

**Intuition.** The three columns of $A_\varepsilon$ are nearly parallel, so subtracting their
projections is a near-total cancellation — and cancellation is where digits die.

**Solution.**

*Step 1 — the conditioning.* Proof 5.11 gives
$\kappa_2(A_\varepsilon) = \sqrt{3+\varepsilon^2}/\varepsilon \approx 1.73 \times 10^{8}$. Note
that $\kappa_2$ is a property of the matrix, defined through $\lVert A \rVert_2 \lVert A^{+}\rVert_2$
as in Definition 3.7: $A_\varepsilon$ is rectangular, so no $A^{-1}$ exists.

*Step 2 — the predicted losses.* By Theorem 4.11, with $u \approx 1.11 \times 10^{-16}$,

$$
u\kappa_2^2 \approx 3.3 \times 10^{0} \ \text{(CGS)}, \qquad
u\kappa_2 \approx 1.9 \times 10^{-8} \ \text{(MGS)}, \qquad
u \ \text{(Householder)} .
$$

The CGS estimate exceeds $1$, which is the bound's way of saying orthogonality is lost outright.

*Step 3 — the mechanism.* $a_2 - (q_1^\top a_2)q_1$ cancels the leading $1$ against itself and
leaves a vector of size $\varepsilon$; CGS forms that difference against the *original* $a_2$, so
the absolute rounding error $O(u)$ is $O(u/\varepsilon)$ relative to the answer, and the effect
compounds over columns.

$$
\boxed{\text{loss: CGS} \approx O(1) \ \gg \ \text{MGS} \approx 10^{-8} \ \gg \ \text{Householder} \approx 10^{-16}}
$$

**Key takeaway.** The gap here is eight orders of magnitude between two algorithms that are
identical on paper, which is Problem L1.12 made quantitative.

In [29]:
eps_l = 1e-8
A_lau = np.array([[1.0, 1.0, 1.0],
                  [eps_l, 0.0, 0.0],
                  [0.0, eps_l, 0.0],
                  [0.0, 0.0, eps_l]])
kap = np.sqrt(3 + eps_l ** 2) / eps_l
u = EPS / 2
print(f"kappa2(A) exact  : {kap:.6e}    numpy: {np.linalg.cond(A_lau):.6e}")
print(f"predicted losses : CGS {u * kap ** 2:.3e}   MGS {u * kap:.3e}   Householder {u:.3e}")
Qc, _ = qr_cgs(A_lau)
Qm, _ = qr_mgs(A_lau)
Qh, _ = np.linalg.qr(A_lau)
for name, Qx in (("CGS", Qc), ("MGS", Qm), ("Householder (numpy)", Qh)):
    print(f"  {name:<22s} ||Q^T Q - I||_2 = {np.linalg.norm(Qx.T @ Qx - np.eye(3), 2):.4e}")
loss_c = np.linalg.norm(Qc.T @ Qc - np.eye(3), 2)
loss_m = np.linalg.norm(Qm.T @ Qm - np.eye(3), 2)
loss_h = np.linalg.norm(Qh.T @ Qh - np.eye(3), 2)
assert loss_c > loss_m > loss_h
assert loss_h < 1e-14

kappa2(A) exact  : 1.732051e+08    numpy: 1.732051e+08
predicted losses : CGS 3.331e+00   MGS 1.923e-08   Householder 1.110e-16
  CGS                    ||Q^T Q - I||_2 = 5.0000e-01
  MGS                    ||Q^T Q - I||_2 = 8.1650e-09
  Householder (numpy)    ||Q^T Q - I||_2 = 2.3773e-16


### Problem L2.8 — Measuring $g$ from drop distances (physics)

**Statement.** A body released from rest falls a distance $s(t) = \tfrac12 g t^2$. Four
measurements give $s = 4.9, 19.7, 44.0, 78.6$ metres at $t = 1,2,3,4$ seconds. Estimate $g$ by
least squares and verify the orthogonality of the residual.

**Intuition.** One unknown and four equations: project the measurement vector onto the single
column $\tfrac12 t^2$.

**Solution.**

*Step 1 — the design matrix.* The model is linear in $g$ with the single column
$x = \tfrac12 (t_1^2, \dots, t_4^2)^\top$, so the normal equations of Theorem 4.7 read
$(x^\top x)\hat{g} = x^\top s$.

*Step 2 — evaluate.* Cancelling the two factors of $\tfrac12$,

$$
\hat{g} = \frac{2 \sum_i t_i^2 s_i}{\sum_i t_i^4}
= \frac{2 (4.9 + 4 \cdot 19.7 + 9 \cdot 44.0 + 16 \cdot 78.6)}{1 + 16 + 81 + 256}
= \frac{3474.6}{354} .
$$

*Step 3 — the value.* $3474.6/354 = 5791/590 = 9.8152542\ldots$ metres per second squared.

*Step 4 — the residual is perpendicular.* Fitted distances $4.9076, 19.6305, 44.1686, 78.5220$
leave $e = (-0.00763, 0.06949, -0.16864, 0.07797)^\top$ with
$\sum_i t_i^2 e_i = 0$, which is $x^\top e = 0$.

$$
\boxed{\hat{g} = \frac{5791}{590} \approx 9.8153 \ \mathrm{m\,s^{-2}}}
$$

**Key takeaway.** Weighting by $t_i^2$ is not a modelling choice; it is what the normal equations
do automatically, and it is why the late, large measurements dominate the estimate.

In [30]:
t_d = np.array([1.0, 2.0, 3.0, 4.0])
s_d = np.array([4.9, 19.7, 44.0, 78.6])
x_d = 0.5 * t_d ** 2
g_hat = (x_d @ s_d) / (x_d @ x_d)
resid_d = s_d - g_hat * x_d
print("g_hat            :", g_hat, "   exact 5791/590 =", 5791 / 590)
print("numerator, denom :", 2 * np.sum(t_d ** 2 * s_d), np.sum(t_d ** 4))
print("fitted distances :", g_hat * x_d)
print("residual         :", resid_d)
print("x^T e            :", x_d @ resid_d)
print("||e||            :", np.linalg.norm(resid_d))
print("relative error against the standard 9.80665 :",
      abs(g_hat - 9.80665) / 9.80665)
assert abs(g_hat - 5791 / 590) < 1e-12
assert abs(x_d @ resid_d) < 1e-12

g_hat            : 9.815254237288135    exact 5791/590 = 9.815254237288135
numerator, denom : 3474.6 354.0
fitted distances : [ 4.9076 19.6305 44.1686 78.522 ]
residual         : [-0.0076  0.0695 -0.1686  0.078 ]
x^T e            : -7.549516567451064e-15
||e||            : 0.19851140939758366
relative error against the standard 9.80665 : 0.0008773880263021382


### Problem L2.9 — Measurement projectors and the Born rule (physics)

**Statement.** A spin-$\tfrac12$ system is in the normalized state $\psi = (0.6, 0.8)^\top$.
Write the measurement projectors for the $z$ and $x$ axes, verify
$\sum_k P_k = I$ and $P_j P_k = 0$ for $j \neq k$, and compute the outcome probabilities and the
expectation values.

**Intuition.** A measurement resolves the state along an orthonormal basis; the Born rule reads
off the squared lengths of the pieces, which sum to one because the pieces are perpendicular.

**Solution.**

*Step 1 — the projectors.* For the $z$ axis the eigenbasis is $e_1, e_2$, giving
$P_{z\pm} = e_1e_1^\top, e_2e_2^\top$. For the $x$ axis it is
$(1,\pm1)^\top/\sqrt2$, giving $P_{x\pm} = \tfrac12\begin{pmatrix} 1 & \pm 1 \\ \pm 1 & 1\end{pmatrix}$.

*Step 2 — completeness and exclusivity.* Each pair satisfies $P_+ + P_- = I$ and $P_+P_- = 0$,
which is Theorem 4.4 for the pair of orthogonal one-dimensional eigenspaces.

*Step 3 — probabilities.* $\lVert P_{z+}\psi \rVert^2 = 0.6^2 = 0.36$ and
$\lVert P_{z-}\psi \rVert^2 = 0.64$. Along $x$, $\langle \psi, (1,1)^\top/\sqrt2\rangle = 1.4/\sqrt2$,
so the probabilities are $1.96/2 = 0.98$ and $0.04/2 = 0.02$.

*Step 4 — expectations.* With $S_z = \tfrac12\operatorname{diag}(1,-1)$ and
$S_x = \tfrac12\begin{pmatrix}0&1\\1&0\end{pmatrix}$ in units of $\hbar$,

$$
\langle S_z \rangle = \tfrac12(0.36 - 0.64) = -0.14,
\qquad
\langle S_x \rangle = \tfrac12(0.98 - 0.02) = 0.48 .
$$

$$
\boxed{(p_{z+}, p_{z-}) = (0.36, 0.64), \quad (p_{x+}, p_{x-}) = (0.98, 0.02), \quad \langle S_z\rangle = -0.14, \ \langle S_x \rangle = 0.48}
$$

**Key takeaway.** Probability conservation in quantum mechanics *is* the Pythagorean identity of
Theorem 4.4; the state collapses to $P_k\psi / \lVert P_k \psi \rVert$, an orthogonal projection
followed by renormalization.

In [31]:
psi = np.array([0.6, 0.8])
basis_z = [np.array([1.0, 0.0]), np.array([0.0, 1.0])]
basis_x = [np.array([1.0, 1.0]) / np.sqrt(2), np.array([1.0, -1.0]) / np.sqrt(2)]
Sz = 0.5 * np.diag([1.0, -1.0])
Sx = 0.5 * np.array([[0.0, 1.0], [1.0, 0.0]])

for axis, basis, op in (("z", basis_z, Sz), ("x", basis_x, Sx)):
    Ps = [np.outer(e, e) for e in basis]
    probs = np.array([np.linalg.norm(P @ psi) ** 2 for P in Ps])
    expect = 0.5 * (probs[0] - probs[1])
    print(f"axis {axis}: probabilities {probs}   sum {probs.sum():.12f}")
    print(f"        P+ + P- = I : {np.allclose(Ps[0] + Ps[1], np.eye(2))}"
          f"    ||P+ P-|| = {np.linalg.norm(Ps[0] @ Ps[1]):.2e}")
    print(f"        <S> from probabilities {expect:+.4f}   from psi^T S psi {psi @ op @ psi:+.4f}")
    assert abs(probs.sum() - 1.0) < 1e-14
    assert abs(expect - psi @ op @ psi) < 1e-14
    collapsed = Ps[0] @ psi / np.linalg.norm(Ps[0] @ psi)
    print(f"        collapsed state {collapsed}   norm {np.linalg.norm(collapsed):.6f}")

axis z: probabilities [0.36 0.64]   sum 1.000000000000
        P+ + P- = I : True    ||P+ P-|| = 0.00e+00
        <S> from probabilities -0.1400   from psi^T S psi -0.1400
        collapsed state [1. 0.]   norm 1.000000
axis x: probabilities [0.98 0.02]   sum 1.000000000000
        P+ + P- = I : True    ||P+ P-|| = 2.47e-32
        <S> from probabilities +0.4800   from psi^T S psi +0.4800
        collapsed state [0.7071 0.7071]   norm 1.000000


### Problem L2.10 — Fourier truncation is an orthogonal projection (physics)

**Statement.** On $[-\pi,\pi]$ with $\langle f,g\rangle = \int_{-\pi}^{\pi} fg\,dt$, expand
$f(t) = t$ in the orthonormal system $e_n(t) = \sin(nt)/\sqrt{\pi}$. Show that the truncation at
$N$ terms is the best $L^2$ approximation by such a combination, and evaluate the residual energy.

**Intuition.** Fourier coefficients are inner products, so truncating is projecting — and Bessel
measures what the discarded harmonics were carrying.

**Solution.**

*Step 1 — orthonormality.* $\int_{-\pi}^{\pi}\sin(mt)\sin(nt)\,dt = \pi\delta_{mn}$, so the
$e_n$ are orthonormal.

*Step 2 — the coefficients.* Integrating by parts,

$$
\int_{-\pi}^{\pi} t \sin(nt)\,dt = \frac{2\pi(-1)^{n+1}}{n},
\qquad
\langle f, e_n\rangle = \frac{2\sqrt{\pi}\,(-1)^{n+1}}{n} .
$$

*Step 3 — the projection.* By Theorem 4.5 the closest element of
$\operatorname{span}(e_1,\dots,e_N)$ is $\sum_{n \le N} \langle f, e_n\rangle e_n$, which is exactly
the truncated Fourier sine series.

*Step 4 — the energy.* $\lVert f \rVert^2 = \int_{-\pi}^{\pi} t^2 dt = \tfrac{2\pi^3}{3}$ and
$\lvert \langle f, e_n\rangle\rvert^2 = 4\pi/n^2$, so the residual energy after $N$ terms is

$$
\frac{2\pi^3}{3} - \sum_{n=1}^{N} \frac{4\pi}{n^2} \ \xrightarrow[N \to \infty]{} \ 0 ,
$$

the limit being Parseval — and, divided by $4\pi$, the statement $\sum_{n\ge1} n^{-2} = \pi^2/6$.

$$
\boxed{\lVert f - P_N f \rVert^2 = \frac{2\pi^3}{3} - \sum_{n=1}^{N}\frac{4\pi}{n^2}, \qquad \sum_{n \ge 1} \frac{1}{n^2} = \frac{\pi^2}{6}}
$$

**Key takeaway.** Bandlimiting a signal is an orthogonal projection, and the energy it removes is
exactly the Bessel deficit — the reason Parseval is called an energy theorem in physics.

In [32]:
nodes_f, wts_f = np.polynomial.legendre.leggauss(600)
tt = np.pi * nodes_f
ww = np.pi * wts_f
f_t = tt


def coeff(n):
    return np.sum(ww * f_t * np.sin(n * tt)) / np.sqrt(np.pi)


energy = np.sum(ww * f_t ** 2)
print("||f||^2 numeric, exact :", energy, 2 * np.pi ** 3 / 3)
for N in (1, 3, 10):
    cs = np.array([coeff(n) for n in range(1, N + 1)])
    approx = sum(cs[n - 1] * np.sin(n * tt) / np.sqrt(np.pi) for n in range(1, N + 1))
    resid_energy = np.sum(ww * (f_t - approx) ** 2)
    bessel = energy - np.sum(cs ** 2)
    print(f"N = {N:2d}  coefficients {np.round(cs, 4)}")
    print(f"        residual energy {resid_energy:.6f}   Bessel deficit {bessel:.6f}")
    perturbed = approx + 0.05 * np.sin(tt) / np.sqrt(np.pi)
    print(f"        perturbing one coefficient gives {np.sum(ww * (f_t - perturbed) ** 2):.6f}"
          f"   (larger, as Theorem 4.5 requires)")
    assert abs(resid_energy - bessel) < 1e-8
    assert np.sum(ww * (f_t - perturbed) ** 2) > resid_energy
print("\nexact coefficient 2 sqrt(pi) (-1)^(n+1)/n for n=1,2,3 :",
      np.array([2 * np.sqrt(np.pi) * (-1) ** (n + 1) / n for n in (1, 2, 3)]))
basel = sum(1 / m ** 2 for m in range(1, 100001))
print("sum 1/n^2 to 1e5, pi^2/6 :", basel, np.pi ** 2 / 6)

||f||^2 numeric, exact : 20.670851120200687 20.670851120199877
N =  1  coefficients [3.5449]
        residual energy 8.104481   Bessel deficit 8.104481
        perturbing one coefficient gives 8.106981   (larger, as Theorem 4.5 requires)
N =  3  coefficients [ 3.5449 -1.7725  1.1816]
        residual energy 3.566624   Bessel deficit 3.566624
        perturbing one coefficient gives 3.569124   (larger, as Theorem 4.5 requires)
N = 10  coefficients [ 3.5449 -1.7725  1.1816 -0.8862  0.709  -0.5908  0.5064 -0.4431  0.3939
 -0.3545]
        residual energy 1.195895   Bessel deficit 1.195895
        perturbing one coefficient gives 1.198395   (larger, as Theorem 4.5 requires)

exact coefficient 2 sqrt(pi) (-1)^(n+1)/n for n=1,2,3 : [ 3.5449 -1.7725  1.1816]
sum 1/n^2 to 1e5, pi^2/6 : 1.6449240668982263 1.6449340668482264


## L3 — Challenge Proofs

### Problem L3.1 — The operator norm as a maximized quadratic form

**Statement.** Let $S \in \mathbb{R}^{n \times n}$ be symmetric. Prove

$$
\lVert S \rVert_2 = \max_{\lVert x \rVert = 1} \lvert x^\top S x \rvert ,
$$

and deduce $\lVert A \rVert_2^2 = \max_{\lVert x \rVert = 1} x^\top A^\top A x$ for any
$A \in \mathbb{R}^{m \times n}$.

**Intuition.** For a symmetric matrix the bilinear form is determined by its diagonal values,
because polarization recovers $y^\top S x$ from quadratic values alone.

**Solution.**

*Step 1 — the easy direction.* For a unit $x$, Cauchy-Schwarz gives
$\lvert x^\top Sx \rvert \le \lVert x \rVert \lVert Sx \rVert \le \lVert S \rVert_2$, so
$s := \max_{\lVert x\rVert = 1}\lvert x^\top Sx\rvert \le \lVert S \rVert_2$.

*Step 2 — polarization.* Symmetry gives $x^\top S y = y^\top S x$, hence

$$
4 y^\top S x = (x+y)^\top S(x+y) - (x-y)^\top S(x-y) .
$$

*Step 3 — bound the right-hand side.* Using $\lvert z^\top S z \rvert \le s \lVert z \rVert^2$ and
the parallelogram law $\lVert x+y\rVert^2 + \lVert x-y \rVert^2 = 2(\lVert x\rVert^2 + \lVert y\rVert^2)$,
for unit $x,y$

$$
4\lvert y^\top Sx\rvert \le s\bigl(\lVert x+y\rVert^2 + \lVert x-y\rVert^2\bigr) = 4s .
$$

*Step 4 — conclude.* Taking $y = Sx/\lVert Sx\rVert$ turns the left side into
$4\lVert Sx\rVert$, so $\lVert Sx \rVert \le s$ for every unit $x$ and
$\lVert S \rVert_2 \le s$.

*Step 5 — the corollary.* $A^\top A$ is symmetric and
$x^\top A^\top A x = \lVert Ax\rVert^2 \ge 0$, so
$\lVert A^\top A\rVert_2 = \max_{\lVert x\rVert = 1}\lVert Ax\rVert^2 = \lVert A\rVert_2^2$.

$$
\boxed{\lVert S \rVert_2 = \max_{\lVert x \rVert = 1} \lvert x^\top S x \rvert, \qquad \lVert A^\top A \rVert_2 = \lVert A \rVert_2^2}
$$

**Key takeaway.** This is the variational characterization of the operator norm, proved without
any eigenvalue theory, and it is Step 1(ii) of Proof 5.10 — the half of Theorem 4.10 that makes
$\kappa_2(A^\top A) = \kappa_2(A)^2$ work.

In [33]:
S_sym = rng.standard_normal((6, 6))
S_sym = (S_sym + S_sym.T) / 2
xs_rand = rng.standard_normal((200000, 6))
xs_rand /= np.linalg.norm(xs_rand, axis=1, keepdims=True)
quad = np.abs(np.einsum("ij,jk,ik->i", xs_rand, S_sym, xs_rand))
print("||S||_2                       :", np.linalg.norm(S_sym, 2))
print("max |x^T S x| over 2e5 samples:", quad.max())
A_rand = rng.standard_normal((7, 4))
print("||A||_2^2, ||A^T A||_2        :", np.linalg.norm(A_rand, 2) ** 2,
      np.linalg.norm(A_rand.T @ A_rand, 2))
assert quad.max() <= np.linalg.norm(S_sym, 2) + 1e-12
assert quad.max() > 0.9 * np.linalg.norm(S_sym, 2)
assert abs(np.linalg.norm(A_rand, 2) ** 2 - np.linalg.norm(A_rand.T @ A_rand, 2)) < 1e-10

||S||_2                       : 3.4079553252591683
max |x^T S x| over 2e5 samples: 3.379292464838867
||A||_2^2, ||A^T A||_2        : 19.21343640868382 19.213436408683826


### Problem L3.2 — Two orthogonal projections are never more than one apart

**Statement.** Let $P, Q \in \mathbb{R}^{n\times n}$ be orthogonal projections. Prove
$\lVert P - Q\rVert_2 \le 1$.

**Intuition.** The difference of two projections and the matrix $I - P - Q$ split the identity
between them, so neither can be large.

**Solution.**

*Step 1 — expand both squares.* Using $P^2 = P$, $Q^2 = Q$,

$$
(P-Q)^2 = P + Q - PQ - QP,
\qquad
(I-P-Q)^2 = I - P - Q + PQ + QP .
$$

*Step 2 — they add to the identity.* Summing the two displays gives

$$
(P-Q)^2 + (I-P-Q)^2 = I .
$$

*Step 3 — the correct positivity argument.* Put $M = I - P - Q$. Symmetry of $P$ and $Q$ makes
$M$ symmetric, so $M^2 = M^\top M$ and therefore

$$
x^\top M^2 x = \lVert Mx \rVert^2 \ge 0 .
$$

Symmetry alone would **not** give this; it is the factorization $M^2 = M^\top M$ that does.

*Step 4 — conclude.* For a unit vector $x$,

$$
\lVert (P-Q)x \rVert^2 = x^\top (P-Q)^2 x = x^\top\bigl(I - M^2\bigr)x = 1 - \lVert Mx\rVert^2 \le 1 .
$$

Maximizing over unit $x$ gives $\lVert P-Q \rVert_2 \le 1$.

$$
\boxed{\lVert P - Q \rVert_2 \le 1}
$$

**Key takeaway.** The bound is attained: for $P$ and $Q$ the projectors onto two orthogonal lines,
$P - Q$ has $\lVert P-Q\rVert_2 = 1$ exactly. The audit's flag was on Step 3 — a symmetric matrix
need not be positive semidefinite, but a square of one always is.

In [34]:
def rand_proj(n, k, rng):
    B, _ = np.linalg.qr(rng.standard_normal((n, k)))
    return B @ B.T


worst = 0.0
for _ in range(300):
    n_p = 6
    Pp = rand_proj(n_p, rng.integers(1, n_p), rng)
    Qp = rand_proj(n_p, rng.integers(1, n_p), rng)
    Mp = np.eye(n_p) - Pp - Qp
    assert np.linalg.norm((Pp - Qp) @ (Pp - Qp) + Mp @ Mp - np.eye(n_p)) < 1e-10
    worst = max(worst, np.linalg.norm(Pp - Qp, 2))
print("largest ||P - Q||_2 over 300 random pairs :", worst)
P_a = np.diag([1.0, 0.0])
Q_a = np.diag([0.0, 1.0])
print("orthogonal lines in R^2, ||P - Q||_2      :", np.linalg.norm(P_a - Q_a, 2))
S_not_psd = np.array([[0.0, 1.0], [1.0, 0.0]])
x_neg = np.array([1.0, -1.0]) / np.sqrt(2)
print("a symmetric matrix that is NOT psd        :", S_not_psd.tolist(),
      "  x^T S x =", x_neg @ S_not_psd @ x_neg)
print("but its square is                        :", (S_not_psd @ S_not_psd).tolist())
assert worst <= 1.0 + 1e-12
assert abs(np.linalg.norm(P_a - Q_a, 2) - 1.0) < 1e-14

largest ||P - Q||_2 over 300 random pairs : 1.0000000000000009
orthogonal lines in R^2, ||P - Q||_2      : 1.0
a symmetric matrix that is NOT psd        : [[0.0, 1.0], [1.0, 0.0]]   x^T S x = -0.9999999999999998
but its square is                        : [[1.0, 0.0], [0.0, 1.0]]


### Problem L3.3 — An oblique projection and its complement have equal norm

**Statement.** Let $P \in \mathbb{R}^{n\times n}$ satisfy $P^2 = P$ with $P \neq 0$ and
$P \neq I$. Prove $\lVert P \rVert_2 = \lVert I - P \rVert_2 = 1/\sin\theta$, where $\theta$ is
the minimal principal angle between $\operatorname{Col}(P)$ and $\operatorname{Null}(P)$.

**Intuition.** A skewed projection stretches by exactly the reciprocal of how far apart its range
and kernel are, and swapping the two subspaces does not change how far apart they are.

**Solution.**

*Step 1 — the two subspaces split the space.* Idempotence gives $x = Px + (I-P)x$ with
$Px \in V := \operatorname{Col}(P)$ and $(I-P)x \in N := \operatorname{Null}(P)$, and
$V \cap N = \{0\}$ because $x \in V$ means $Px = x$. So $\mathbb{R}^n = V \oplus N$, the sum being
direct but not orthogonal.

*Step 2 — express the norm as a distance.* Writing $x = r + w$ with $r \in V$, $w \in N$ gives
$Px = r$, hence

$$
\lVert P \rVert_2 = \max_{r \in V,\, \lVert r \rVert = 1} \ \frac{1}{\min_{w \in N} \lVert r + w\rVert}
= \frac{1}{\min_{r \in V, \lVert r\rVert = 1} \operatorname{dist}(r, N)} .
$$

*Step 3 — the distance is a sine.* By Theorem 4.5,
$\operatorname{dist}(r,N)^2 = \lVert r\rVert^2 - \lVert P_N r\rVert^2 = 1 - \lVert P_N r\rVert^2$
with $P_N$ the *orthogonal* projector onto $N$. And

$$
\max_{r \in V, \lVert r\rVert = 1} \lVert P_N r \rVert
= \max_{r \in V,\, w \in N,\ \lVert r\rVert = \lVert w\rVert = 1} \langle r, w \rangle =: \cos\theta ,
$$

using $\lVert P_N r\rVert = \max_{\lVert w \rVert = 1, w \in N}\langle r, w\rangle$. Therefore
$\min_r \operatorname{dist}(r,N) = \sin\theta$ and $\lVert P \rVert_2 = 1/\sin\theta$.

*Step 4 — the symmetry.* The defining expression for $\cos\theta$ is symmetric in $V$ and $N$.
Since $\operatorname{Col}(I-P) = N$ and $\operatorname{Null}(I-P) = V$, Step 3 applied to $I-P$
gives the same angle, hence the same norm.

*Step 5 — non-degeneracy.* $P \neq 0, I$ makes both $V$ and $N$ non-trivial, and
$V \cap N = \{0\}$ makes $\cos\theta \lt 1$, so $\sin\theta \gt 0$.

$$
\boxed{\lVert P \rVert_2 = \lVert I - P \rVert_2 = \frac{1}{\sin\theta}}
$$

**Key takeaway.** For an *orthogonal* projection $\theta = \pi/2$ and the norm is $1$; every
excess over $1$ measures how skew the projection is. Reference: Szyld, *Numerical Algorithms*
**42** (2006), 309-323.

In [35]:
def oblique_projector(V, N):
    """The projection onto span(V) along span(N), for complementary subspaces."""
    Bmat = np.hstack([V, N])
    k = V.shape[1]
    E = np.zeros((Bmat.shape[1], Bmat.shape[1]))
    E[:k, :k] = np.eye(k)
    return Bmat @ E @ np.linalg.inv(Bmat)


for tilt in (1.0, 0.5, 0.2, 0.05):
    Vs = np.array([[1.0], [0.0]])
    Ns = np.array([[1.0], [tilt]])
    Ns = Ns / np.linalg.norm(Ns)
    P_ob = oblique_projector(Vs, Ns)
    cos_t = abs(Vs[:, 0] @ Ns[:, 0])
    sin_t = np.sqrt(1 - cos_t ** 2)
    print(f"tilt {tilt:5.2f}:  ||P||_2 = {np.linalg.norm(P_ob, 2):9.4f}"
          f"   ||I-P||_2 = {np.linalg.norm(np.eye(2) - P_ob, 2):9.4f}"
          f"   1/sin(theta) = {1 / sin_t:9.4f}")
    assert np.allclose(P_ob @ P_ob, P_ob)
    assert abs(np.linalg.norm(P_ob, 2) - np.linalg.norm(np.eye(2) - P_ob, 2)) < 1e-9
    assert abs(np.linalg.norm(P_ob, 2) - 1 / sin_t) < 1e-9

tilt  1.00:  ||P||_2 =    1.4142   ||I-P||_2 =    1.4142   1/sin(theta) =    1.4142
tilt  0.50:  ||P||_2 =    2.2361   ||I-P||_2 =    2.2361   1/sin(theta) =    2.2361
tilt  0.20:  ||P||_2 =    5.0990   ||I-P||_2 =    5.0990   1/sin(theta) =    5.0990
tilt  0.05:  ||P||_2 =   20.0250   ||I-P||_2 =   20.0250   1/sin(theta) =   20.0250


### Problem L3.4 — Von Neumann's alternating projections

**Statement.** Let $U, W \subseteq \mathbb{R}^n$ be subspaces with orthogonal projectors
$P_U, P_W$, and set $T = P_U P_W$. Prove that for every $x$,

$$
\lim_{k \to \infty} T^k x = P_{U \cap W}\, x .
$$

**Intuition.** Bouncing between two subspaces can never leave their intersection, and outside the
intersection every bounce is strictly contracting.

**Solution.**

*Step 1 — $T$ is non-expansive.* For any $y$, Theorem 4.4 gives
$\lVert P_U y\rVert \le \lVert y \rVert$, so $\lVert Tx\rVert \le \lVert P_W x\rVert \le \lVert x\rVert$.

*Step 2 — the fixed points are $M := U \cap W$.* If $z \in M$ then $P_Wz = z$ and $P_Uz = z$, so
$Tz = z$. Conversely if $\lVert Tz \rVert = \lVert z \rVert$ then both inequalities of Step 1 are
equalities. Now $\lVert P_W z\rVert = \lVert z\rVert$ together with
$\lVert z\rVert^2 = \lVert P_Wz\rVert^2 + \lVert z - P_Wz\rVert^2$ forces $P_Wz = z$, that is
$z \in W$; the same argument then gives $z \in U$. In particular $Tz = z$ implies $z \in M$.

*Step 3 — $M^{\perp}$ is invariant under $T$.* $T^\top = P_W P_U$, and for $z \in M$ we have
$P_Uz = P_Wz = z$, hence $T^\top z = z$. So for $x \in M^{\perp}$ and $z \in M$,

$$
\langle Tx, z\rangle = \langle x, T^\top z\rangle = \langle x, z \rangle = 0 ,
$$

so $Tx \in M^{\perp}$.

*Step 4 — strict contraction on $M^{\perp}$.* If $z \in M^{\perp}$ and $\lVert Tz\rVert = \lVert z\rVert$
then $z \in M$ by Step 2, so $z \in M \cap M^{\perp} = \{0\}$. Hence
$\lVert Tz \rVert \lt \lVert z\rVert$ for every non-zero $z \in M^{\perp}$.

*Step 5 — a uniform rate.* The unit sphere of $M^{\perp}$ is compact and $z \mapsto \lVert Tz\rVert$
is continuous, so $c := \max\{\lVert Tz\rVert : z \in M^{\perp},\ \lVert z\rVert = 1\}$ is attained
and $c \lt 1$. Invariance from Step 3 then gives $\lVert T^kz\rVert \le c^k \lVert z\rVert$.

*Step 6 — assemble.* Split $x = m + z$ with $m \in M$, $z \in M^{\perp}$. Then
$T^kx = m + T^kz \to m = P_M x$, geometrically at rate $c$.

$$
\boxed{\lim_{k\to\infty} (P_U P_W)^k = P_{U \cap W}, \qquad \lVert T^k x - P_{U\cap W}x \rVert \le c^k \lVert x \rVert, \ c \lt 1}
$$

**Key takeaway.** The contraction factor $c$ is $\cos^2$ of the principal angle between $U$ and
$W$: nearly parallel subspaces make alternating projections converge slowly, which is why the
method is fast for well-separated constraints and slow for nearly redundant ones.

In [36]:
n_v = 6
Ub, _ = np.linalg.qr(rng.standard_normal((n_v, 4)))
common = Ub[:, :2]
extra, _ = np.linalg.qr(rng.standard_normal((n_v, 2)))
Wb, _ = np.linalg.qr(np.hstack([common, extra[:, :1]]))
P_U = Ub @ Ub.T
P_W = Wb @ Wb.T
P_M = common @ common.T
T = P_U @ P_W

x0 = rng.standard_normal(n_v)
target = P_M @ x0
errs = []
xk = x0.copy()
for k in range(121):
    errs.append(np.linalg.norm(xk - target))
    xk = T @ xk
errs = np.array(errs)
print("iterate errors  :", np.array2string(errs[:8], precision=5))
print("error at k = 120:", errs[-1])
ratios = errs[5:12] / errs[4:11]
print("error ratios    :", np.round(ratios, 5), "  all below 1:", np.all(ratios < 1))
print("||T^120 - P_M|| :", np.linalg.norm(np.linalg.matrix_power(T, 120) - P_M))
assert errs[-1] < 1e-8
assert np.all(ratios < 1.0)

iterate errors  : [1.26399 0.23613 0.19272 0.15729 0.12837 0.10477 0.0855  0.06978]
error at k = 120: 7.46911997906146e-12
error ratios    : [0.8161 0.8161 0.8161 0.8161 0.8161 0.8161 0.8161]   all below 1: True
||T^120 - P_M|| : 2.8576124795701015e-11


### Problem L3.5 — A product of orthogonal projections, and when it is one

**Statement.** Let $P_1, P_2$ be the orthogonal projections onto $W_1, W_2$. Prove that
$P_1P_2$ is an orthogonal projection if and only if $P_1P_2 = P_2P_1$, and identify the subspace.

**Intuition.** A projector must be symmetric, and the transpose of a product reverses the order —
so symmetry of the product *is* commutativity.

**Solution.**

*Step 1 — necessity.* If $M = P_1P_2$ is an orthogonal projection then $M^\top = M$, and

$$
M^\top = (P_1P_2)^\top = P_2^\top P_1^\top = P_2 P_1 ,
$$

so $P_1P_2 = P_2P_1$.

*Step 2 — sufficiency, symmetry.* If the two commute then
$M^\top = P_2P_1 = P_1P_2 = M$.

*Step 3 — sufficiency, idempotence.*

$$
M^2 = P_1(P_2P_1)P_2 = P_1(P_1P_2)P_2 = P_1^2 P_2^2 = P_1P_2 = M .
$$

By Theorem 4.6, $M$ is the orthogonal projector onto $\operatorname{Col}(M)$.

*Step 4 — the subspace.* $Mx = P_1(P_2x) \in W_1$ and $Mx = P_2(P_1x) \in W_2$, so
$\operatorname{Col}(M) \subseteq W_1 \cap W_2$. Conversely $w \in W_1 \cap W_2$ has
$Mw = P_1P_2w = P_1w = w$, so $w \in \operatorname{Col}(M)$.

$$
\boxed{P_1P_2 \text{ is an orthogonal projection} \iff P_1P_2 = P_2P_1, \text{ and then } P_1P_2 = P_{W_1 \cap W_2}}
$$

**Key takeaway.** When the projectors do not commute the product is not a projector, and
Problem L3.4 is what one has to do instead: iterate until the limit is one.

In [37]:
Wa, _ = np.linalg.qr(rng.standard_normal((5, 2)))
Wc, _ = np.linalg.qr(rng.standard_normal((5, 2)))
P_a2, P_c2 = Wa @ Wa.T, Wc @ Wc.T
print("non-commuting pair:")
print("  ||P1 P2 - P2 P1||    :", np.linalg.norm(P_a2 @ P_c2 - P_c2 @ P_a2))
print("  ||(P1P2)^2 - P1P2||  :", np.linalg.norm((P_a2 @ P_c2) @ (P_a2 @ P_c2) - P_a2 @ P_c2))

E1 = np.diag([1.0, 1.0, 1.0, 0.0, 0.0])
E2 = np.diag([0.0, 1.0, 1.0, 1.0, 0.0])
print("commuting pair (coordinate projectors):")
print("  ||P1 P2 - P2 P1||    :", np.linalg.norm(E1 @ E2 - E2 @ E1))
print("  product              :", np.diag(E1 @ E2), " <- projector onto the intersection")
print("  ||(P1P2)^2 - P1P2||  :", np.linalg.norm((E1 @ E2) @ (E1 @ E2) - E1 @ E2))
assert np.linalg.norm(P_a2 @ P_c2 - P_c2 @ P_a2) > 1e-3
assert np.linalg.norm((P_a2 @ P_c2) @ (P_a2 @ P_c2) - P_a2 @ P_c2) > 1e-3
assert np.allclose((E1 @ E2) @ (E1 @ E2), E1 @ E2)

non-commuting pair:
  ||P1 P2 - P2 P1||    : 0.8322844499656084
  ||(P1P2)^2 - P1P2||  : 0.42114296205868623
commuting pair (coordinate projectors):
  ||P1 P2 - P2 P1||    : 0.0
  product              : [0. 1. 1. 0. 0.]  <- projector onto the intersection
  ||(P1P2)^2 - P1P2||  : 0.0


### Problem L3.6 — Parseval's identity in a Hilbert space

**Statement.** Let $\mathcal{H}$ be a complex Hilbert space with a complete orthonormal system
$\{q_1, q_2, \dots\}$, meaning the closed span of the $q_i$ is $\mathcal{H}$. Prove that for all
$x, y$,

$$
\langle x, y\rangle = \sum_{i \ge 1} \langle x, q_i\rangle \overline{\langle y, q_i\rangle} .
$$

**Intuition.** Once both vectors are written in the same orthonormal coordinates, the inner
product is the usual coordinate sum — the conjugate is what keeps the formula consistent with
conjugate symmetry.

**Solution.**

*Step 1 — the expansions converge.* Bessel (Theorem 4.2) makes
$\sum_i \lvert\langle x,q_i\rangle\rvert^2$ finite, so the partial sums
$x_N = \sum_{i\le N}\langle x, q_i\rangle q_i$ are Cauchy and converge in $\mathcal{H}$.
Completeness of the system forces $x_N \to x$.

*Step 2 — the inner product is continuous.* Cauchy-Schwarz gives

$$
\lvert \langle x_N, y_N\rangle - \langle x, y\rangle \rvert
\le \lVert x_N - x\rVert \lVert y_N \rVert + \lVert x \rVert \lVert y_N - y\rVert \to 0 .
$$

*Step 3 — evaluate the finite inner products.* Orthonormality collapses the double sum:

$$
\langle x_N, y_N\rangle = \sum_{i \le N}\sum_{j \le N} \langle x,q_i\rangle \overline{\langle y,q_j\rangle}\langle q_i,q_j\rangle
= \sum_{i \le N} \langle x, q_i\rangle\overline{\langle y, q_i\rangle} .
$$

*Step 4 — pass to the limit* using Step 2.

$$
\boxed{\langle x, y\rangle = \sum_{i \ge 1}\langle x, q_i\rangle \overline{\langle y, q_i\rangle}, \qquad \lVert x\rVert^2 = \sum_{i\ge1}\lvert\langle x,q_i\rangle\rvert^2}
$$

**Key takeaway.** The conjugate on the second factor is not decoration: without it the right-hand
side would fail conjugate symmetry and the identity would be false in a complex space. Over
$\mathbb{R}$ it disappears.

In [38]:
m_dim = 12
Qc_basis, _ = np.linalg.qr(rng.standard_normal((m_dim, m_dim)) + 1j * rng.standard_normal((m_dim, m_dim)))
x_h = rng.standard_normal(m_dim) + 1j * rng.standard_normal(m_dim)
y_h = rng.standard_normal(m_dim) + 1j * rng.standard_normal(m_dim)
cx = Qc_basis.conj().T @ x_h
cy = Qc_basis.conj().T @ y_h
lhs = np.vdot(y_h, x_h)                      # <x, y> = sum x_i conj(y_i)
rhs = np.sum(cx * np.conj(cy))
print("||Q^* Q - I||        :", np.linalg.norm(Qc_basis.conj().T @ Qc_basis - np.eye(m_dim)))
print("<x,y>  direct        :", lhs)
print("<x,y>  by coefficients:", rhs)
print("with the conjugate dropped (wrong):", np.sum(cx * cy))
print("Parseval for ||x||^2 :", np.linalg.norm(x_h) ** 2, np.sum(np.abs(cx) ** 2))
assert abs(lhs - rhs) < 1e-10
assert abs(np.linalg.norm(x_h) ** 2 - np.sum(np.abs(cx) ** 2)) < 1e-10

||Q^* Q - I||        : 1.618456436853406e-15
<x,y>  direct        : (-1.6816845401780531-2.8047772255829706j)
<x,y>  by coefficients: (-1.6816845401780522-2.8047772255829773j)
with the conjugate dropped (wrong): (-1.9264778087810184-0.853579882809147j)
Parseval for ||x||^2 : 19.511397409125532 19.511397409125536


### Problem L3.7 — Every square matrix is orthogonally similar to a Hessenberg matrix

**Statement.** For $A \in \mathbb{R}^{n\times n}$ construct an orthogonal $Q$ with
$H = Q^\top A Q$ upper Hessenberg, that is $H_{ij} = 0$ whenever $i \gt j+1$.

**Intuition.** Reflectors that leave the first $k$ coordinates alone can clear a column below the
subdiagonal without the matching right-multiplication filling it back in.

**Solution.**

*Step 1 — the step.* At stage $k \in \{1,\dots,n-2\}$ let
$x = (A_{k+1,k}, \dots, A_{n,k})^\top \in \mathbb{R}^{n-k}$ and let $\tilde{H}_k$ be the reflector
of Proof 5.8 Part A sending $x$ to a multiple of $e_1$. Embed it as

$$
H_k = \begin{bmatrix} I_k & 0 \\ 0 & \tilde{H}_k \end{bmatrix},
$$

which is orthogonal and symmetric.

*Step 2 — left multiplication clears the column.* $H_k A$ replaces rows $k+1,\dots,n$ and turns
entries $A_{k+2,k}, \dots, A_{n,k}$ into zeros.

*Step 3 — right multiplication does not refill it.* $H_k$ acts as the identity on the first $k$
coordinates, so post-multiplying by $H_k^\top = H_k$ mixes only columns $k+1,\dots,n$ and leaves
column $k$ untouched. This is exactly why the similarity transform must start one row *below* the
diagonal: clearing the diagonal entry too would destroy the zeros already created.

*Step 4 — iterate.* After $n-2$ stages, $Q = H_1H_2\cdots H_{n-2}$ is orthogonal and
$Q^\top A Q$ is upper Hessenberg.

$$
\boxed{Q^\top A Q = H \text{ upper Hessenberg}, \quad Q \text{ orthogonal}, \quad O(n^3) \text{ flops}}
$$

**Key takeaway.** Hessenberg form is as far as orthogonal similarity can go in finitely many steps;
reaching triangular form requires an iteration, which is the subject of the numerical eigenvalue
algorithms downstream of this module.

In [39]:
def to_hessenberg(A):
    """Orthogonal reduction to upper Hessenberg form by Householder reflectors."""
    n = A.shape[0]
    Hm = A.astype(float).copy()
    Qacc = np.eye(n)
    for k in range(n - 2):
        x = Hm[k + 1:, k].copy()
        if np.linalg.norm(x) == 0.0:
            continue
        s = 1.0 if x[0] >= 0 else -1.0
        v = x.copy()
        v[0] = x[0] + s * np.linalg.norm(x)
        v = v / np.linalg.norm(v)
        Hm[k + 1:, :] -= 2.0 * np.outer(v, v @ Hm[k + 1:, :])
        Hm[:, k + 1:] -= 2.0 * np.outer(Hm[:, k + 1:] @ v, v)
        Qacc[:, k + 1:] -= 2.0 * np.outer(Qacc[:, k + 1:] @ v, v)
    return Qacc, Hm


A_hess = rng.standard_normal((6, 6))
Qh, Hh = to_hessenberg(A_hess)
below = np.tril(Hh, -2)
print("H (rounded)      :\n", np.round(Hh, 4))
print("max |entry below the subdiagonal| :", np.abs(below).max())
print("||Q^T Q - I||                     :", np.linalg.norm(Qh.T @ Qh - np.eye(6)))
print("||Q^T A Q - H||                   :", np.linalg.norm(Qh.T @ A_hess @ Qh - Hh))
assert np.abs(below).max() < 1e-12
assert np.linalg.norm(Qh.T @ Qh - np.eye(6)) < 1e-12
assert np.linalg.norm(Qh.T @ A_hess @ Qh - Hh) < 1e-11

H (rounded)      :
 [[ 1.5634  0.4616  0.6111  0.1455 -0.7296 -1.2483]
 [ 2.2855  0.7699 -0.0261  0.0778  0.9397  0.011 ]
 [ 0.     -1.3288  0.1062  0.3052 -0.3143 -0.2241]
 [ 0.      0.      0.5775 -1.6146 -0.921  -0.8331]
 [ 0.      0.      0.      2.3876 -1.2034  0.5093]
 [ 0.      0.      0.     -0.     -0.3697 -0.1832]]
max |entry below the subdiagonal| : 4.440892098500626e-16
||Q^T Q - I||                     : 7.498676947801267e-16
||Q^T A Q - H||                   : 1.8100346998592785e-15


### Problem L3.8 — Rank-revealing QR with column pivoting

**Statement.** Let $A \in \mathbb{R}^{m\times n}$ have rank $r$. Prove that there are a
permutation $\Pi$, an orthogonal $Q$, and an upper triangular block $R_{11} \in \mathbb{R}^{r\times r}$
with strictly positive, non-increasing diagonal such that

$$
A\Pi = Q \begin{bmatrix} R_{11} & R_{12} \\ 0 & 0 \end{bmatrix} .
$$

**Intuition.** Always eliminate the largest remaining column first; the process runs out of
non-zero columns after exactly $r$ steps, and that is where the rank shows up.

**Solution.**

*Step 1 — the greedy step.* At stage $k$, among columns $k, \dots, n$ of the current matrix
compute the norm of the part lying in rows $k, \dots, m$, and swap the largest to position $k$.
If that largest norm is zero, stop.

*Step 2 — apply a reflector.* If it is non-zero, apply the reflector of Proof 5.8 Part A to rows
$k,\dots,m$. This makes $\lvert r_{kk}\rvert$ equal to that largest norm, which is strictly
positive.

*Step 3 — the diagonal decreases.* After stage $k$ the trailing norm of every remaining column is
at most its norm over rows $k,\dots,m$, which was at most $\lvert r_{kk}\rvert$ by the choice in
Step 1. Hence $\lvert r_{11}\rvert \ge \lvert r_{22}\rvert \ge \cdots$.

*Step 4 — the process stops at $r$.* Say it performs $s$ non-zero stages. The final matrix is
$\begin{bmatrix} R_{11} & R_{12} \\ 0 & 0\end{bmatrix}$ with $R_{11}$ upper triangular of size
$s$ and non-zero diagonal, so its rank is $s$. Multiplying by the invertible $Q$ and $\Pi$ does
not change rank, so $s = \operatorname{rank}(A) = r$.

*Step 5 — signs.* Multiplying on the left by a diagonal matrix of $\pm 1$ — orthogonal — makes
every $r_{kk}$ positive without disturbing the zero blocks.

$$
\boxed{A\Pi = Q\begin{bmatrix} R_{11} & R_{12} \\ 0 & 0\end{bmatrix}, \quad r_{11} \ge r_{22} \ge \cdots \ge r_{rr} \gt 0}
$$

**Key takeaway.** Plain QR cannot see rank — Section 7.4 of the theory notebook shows its
factorization becoming non-unique instead. Pivoting turns the diagonal of $R$ into a rank
detector, at the cost of the column ordering. The sharper question of how well
$\lvert r_{kk}\rvert$ tracks the singular values belongs to
[Module 07](../07_canonical_forms_and_svd/first_principles.ipynb).

In [40]:
from scipy.linalg import qr as scipy_qr

B_rank = rng.standard_normal((8, 3))
C_mix = rng.standard_normal((3, 6))
A_rr = B_rank @ C_mix                        # 8 x 6 of rank 3
Qr, Rr, piv = scipy_qr(A_rr, pivoting=True)
Pi = np.zeros((6, 6))
Pi[piv, np.arange(6)] = 1.0
print("true rank                :", np.linalg.matrix_rank(A_rr))
print("|diag(R)|                :", np.abs(np.diag(Rr)))
print("non-increasing           :", np.all(np.diff(np.abs(np.diag(Rr))) <= 1e-12))
print("||A Pi - Q R||           :", np.linalg.norm(A_rr @ Pi - Qr @ Rr))
print("||rows 4..6 of R||       :", np.linalg.norm(Rr[3:, :]))
assert np.linalg.matrix_rank(A_rr) == 3
assert np.linalg.norm(A_rr @ Pi - Qr @ Rr) < 1e-11
assert np.linalg.norm(Rr[3:, :]) < 1e-11
assert np.all(np.diff(np.abs(np.diag(Rr)[:3])) <= 1e-12)

true rank                : 3
|diag(R)|                : [7.4704 3.55   1.893  0.     0.     0.    ]
non-increasing           : True
||A Pi - Q R||           : 2.311224630742853e-15
||rows 4..6 of R||       : 1.293391955168561e-15


### Problem L3.9 — The Cayley transform of a skew-symmetric matrix

**Statement.** Let $A \in \mathbb{R}^{n\times n}$ satisfy $A^\top = -A$. Show that $I + A$ is
invertible, that $Q = (I-A)(I+A)^{-1}$ is orthogonal, and that $\det Q = 1$.

**Intuition.** Skew-symmetric matrices are infinitesimal rotations; the Cayley transform is the
rational way to exponentiate them into finite ones.

**Solution.**

*Step 1 — invertibility.* If $(I+A)x = 0$ then $x^\top x = -x^\top A x$. Skewness gives
$x^\top A x = (x^\top A x)^\top = x^\top A^\top x = -x^\top A x$, so $x^\top A x = 0$ and hence
$\lVert x\rVert^2 = 0$.

*Step 2 — the transpose.* $Q^\top = \bigl((I+A)^{-1}\bigr)^\top (I-A)^\top = (I-A)^{-1}(I+A)$.

*Step 3 — orthogonality.* $(I+A)$ and $(I-A)$ commute, since both are polynomials in $A$, and so
do their inverses. Hence

$$
Q^\top Q = (I-A)^{-1}(I+A)(I-A)(I+A)^{-1} = (I-A)^{-1}(I-A)(I+A)(I+A)^{-1} = I .
$$

*Step 4 — the determinant.* $\det(I-A) = \det\bigl((I-A)^\top\bigr) = \det(I+A)$, so
$\det Q = \det(I-A)/\det(I+A) = 1$.

$$
\boxed{Q^\top Q = I, \qquad \det Q = 1}
$$

**Key takeaway.** The image is exactly the rotations without $-1$ as an eigenvalue, so the Cayley
transform is a chart on $\mathrm{SO}(n)$ — the standard trick for optimizing over orthogonal
weight matrices without leaving the manifold.

In [41]:
K = rng.standard_normal((5, 5))
K = (K - K.T) / 2
Q_cay = (np.eye(5) - K) @ np.linalg.inv(np.eye(5) + K)
print("||K^T + K||        :", np.linalg.norm(K.T + K))
print("||Q^T Q - I||      :", np.linalg.norm(Q_cay.T @ Q_cay - np.eye(5)))
print("det Q              :", np.linalg.det(Q_cay))
print("||Q x|| - ||x||    :", abs(np.linalg.norm(Q_cay @ np.ones(5)) - np.sqrt(5)))
assert np.linalg.norm(Q_cay.T @ Q_cay - np.eye(5)) < 1e-10
assert abs(np.linalg.det(Q_cay) - 1.0) < 1e-10

||K^T + K||        : 0.0
||Q^T Q - I||      : 6.095862607652694e-16
det Q              : 1.0
||Q x|| - ||x||    : 4.440892098500626e-16


### Problem L3.10 — The inverse Cayley transform

**Statement.** Let $Q$ be orthogonal with $Q - I$ invertible. Show that
$S = (Q+I)(Q-I)^{-1}$ is skew-symmetric.

**Intuition.** This is Problem L3.9 read backwards, so the same commuting-polynomial argument
runs in reverse.

**Solution.**

*Step 1 — transpose.* $S^\top = \bigl((Q-I)^{-1}\bigr)^\top (Q+I)^\top = (Q^\top - I)^{-1}(Q^\top + I)$.

*Step 2 — rewrite with $Q^\top = Q^{-1}$.*

$$
Q^\top - I = Q^{-1}(I - Q) = -Q^{-1}(Q - I),
\qquad
Q^\top + I = Q^{-1}(I + Q) = (Q+I)Q^{-1} .
$$

*Step 3 — invert the first.* $(Q^\top - I)^{-1} = -(Q-I)^{-1}Q$.

*Step 4 — combine.* Substituting into Step 1 and using that $Q$ commutes with $Q+I$,

$$
S^\top = -(Q-I)^{-1} Q (Q+I) Q^{-1} = -(Q-I)^{-1}(Q+I) = -S ,
$$

the last equality because $(Q-I)^{-1}$ and $(Q+I)$ are both polynomials in $Q$ (or its inverse)
and therefore commute.

*Step 5 — how the two transforms fit together.* If $Q = (I-A)(I+A)^{-1}$ as in Problem L3.9, then
$Q + I = 2(I+A)^{-1}$ and $Q - I = -2A(I+A)^{-1}$, so this problem's $S$ equals $-A^{-1}$. The
transform that actually inverts Problem L3.9 is $A = (I-Q)(I+Q)^{-1}$.

$$
\boxed{S^\top = -S; \quad \text{and } (I-Q)(I+Q)^{-1} \text{ inverts } Q = (I-A)(I+A)^{-1}}
$$

**Key takeaway.** Together with Problem L3.9 this gives a bijection between skew-symmetric
matrices and orthogonal matrices without eigenvalue $-1$ — a rational parameterization of
rotations that avoids the matrix exponential entirely.

In [42]:
K4 = rng.standard_normal((4, 4))
K4 = (K4 - K4.T) / 2
I4 = np.eye(4)
Q_in = (I4 - K4) @ np.linalg.inv(I4 + K4)          # orthogonal, and Q - I invertible
print("||Q^T Q - I||               :", np.linalg.norm(Q_in.T @ Q_in - I4))
print("kappa2(Q - I)               :", np.linalg.cond(Q_in - I4))
S_cay = (Q_in + I4) @ np.linalg.inv(Q_in - I4)
print("||S^T + S||                 :", np.linalg.norm(S_cay.T + S_cay))
print("||S + K^-1||                :", np.linalg.norm(S_cay + np.linalg.inv(K4)))
K_back = (I4 - Q_in) @ np.linalg.inv(I4 + Q_in)
print("||(I-Q)(I+Q)^-1 - K||       :", np.linalg.norm(K_back - K4))
assert np.linalg.norm(S_cay.T + S_cay) < 1e-8
assert np.linalg.norm(K_back - K4) < 1e-9

||Q^T Q - I||               : 7.240456418871298e-16
kappa2(Q - I)               : 1.6249304666106796
||S^T + S||                 : 8.561666272932958e-16
||S + K^-1||                : 7.230563753411894e-16
||(I-Q)(I+Q)^-1 - K||       : 3.8546691384007913e-16


### Problem L3.11 — Projection onto a closed convex set is non-expansive

**Statement.** Let $S \subseteq \mathbb{R}^n$ be closed and convex and let
$P_S(x) = \arg\min_{y\in S}\lVert x - y\rVert$. Prove that $P_S$ is well defined and that

$$
\lVert P_S(x) - P_S(y)\rVert \le \lVert x - y\rVert \qquad \text{for all } x, y .
$$

**Intuition.** For a subspace the error is exactly perpendicular; for a convex set it can only
lean *away*, and that one-sided inequality is enough.

**Solution.**

*Step 1 — existence.* Intersecting $S$ with a closed ball containing at least one point of $S$
gives a non-empty compact set, on which the continuous function
$y \mapsto \lVert x - y\rVert$ attains its minimum.

*Step 2 — uniqueness.* If $p_1 \neq p_2$ both attain the minimum $d$, convexity puts the midpoint
$m = \tfrac12(p_1+p_2)$ in $S$, and the parallelogram law gives

$$
\lVert x - m\rVert^2 = d^2 - \tfrac14 \lVert p_1 - p_2\rVert^2 \lt d^2 ,
$$

a contradiction.

*Step 3 — the variational inequality.* Let $p = P_S(x)$ and $z \in S$. Convexity puts
$p + t(z-p)$ in $S$ for $t \in (0,1]$, so

$$
\lVert x - p - t(z-p)\rVert^2 \ge \lVert x - p\rVert^2 ,
$$

which expands to $-2t\langle x-p, z-p\rangle + t^2\lVert z-p\rVert^2 \ge 0$. Dividing by $t$ and
letting $t \to 0^{+}$ gives

$$
\langle x - P_S(x),\ z - P_S(x)\rangle \le 0 \qquad \text{for every } z \in S .
$$

*Step 4 — apply it twice.* Take $z = P_S(y)$ in the inequality for $x$, and $z = P_S(x)$ in the
one for $y$; adding the two and writing $\delta = P_S(x) - P_S(y)$,

$$
\lVert \delta \rVert^2 \le \langle x - y, \ \delta \rangle .
$$

*Step 5 — Cauchy-Schwarz.* The right side is at most $\lVert x-y\rVert \lVert \delta\rVert$, and
dividing by $\lVert \delta \rVert$ (trivial if it vanishes) gives the claim.

$$
\boxed{\lVert P_S(x) - P_S(y)\rVert \le \lVert x - y \rVert}
$$

**Key takeaway.** Step 3 is the convex replacement for "the error is perpendicular": equality
holds when $S$ is a subspace, because then $\pm(z-p)$ are both admissible directions. This is why
projected gradient descent inherits the convergence rate of the unprojected method.

In [43]:
radius = 1.5


def proj_ball(x):
    nx = np.linalg.norm(x)
    return x if nx <= radius else radius * x / nx


worst_ratio = 0.0
for _ in range(4000):
    xa = 3.0 * rng.standard_normal(4)
    xb = 3.0 * rng.standard_normal(4)
    pa, pb = proj_ball(xa), proj_ball(xb)
    ratio = np.linalg.norm(pa - pb) / np.linalg.norm(xa - xb)
    worst_ratio = max(worst_ratio, ratio)
    vi = (xa - pa) @ (pb - pa)
    assert vi <= 1e-10
print("largest contraction ratio over 4000 pairs :", worst_ratio)
print("variational inequality <x - P(x), z - P(x)> <= 0 held in every case")
assert worst_ratio <= 1.0 + 1e-12

largest contraction ratio over 4000 pairs : 0.9389211559226237
variational inequality <x - P(x), z - P(x)> <= 0 held in every case


### Problem L3.12 — Two trace inequalities for positive definite matrices

**Statement.** Let $A, B \in \mathbb{R}^{n\times n}$ be symmetric positive definite and let $P$ be
an orthogonal projection. Prove $\operatorname{tr}(AB) \gt 0$ and $\operatorname{tr}(PAP) \le \operatorname{tr}(A)$.

**Intuition.** Cholesky turns "positive definite" into "a product $LL^\top$", after which both
statements are sums of squared lengths.

**Solution.**

*Step 1 — trace cyclicity.* As in Problem L0.7, $\operatorname{tr}(XY) = \operatorname{tr}(YX)$.

*Step 2 — factor $A$.* Positive definiteness gives a Cholesky factorization $A = LL^\top$ with
$L$ invertible.

*Step 3 — the first inequality.*

$$
\operatorname{tr}(AB) = \operatorname{tr}(LL^\top B) = \operatorname{tr}(L^\top B L)
= \sum_{i=1}^{n} (Le_i)^\top B\,(Le_i) \ \gt \ 0 ,
$$

since $L$ invertible makes every $Le_i \neq 0$ and $B$ is positive definite.

*Step 4 — bound the projector.* $P$ orthogonal projection gives
$y^\top P y = y^\top P^\top P y = \lVert Py\rVert^2$, and Theorem 4.4 gives
$\lVert y\rVert^2 = \lVert Py\rVert^2 + \lVert (I-P)y\rVert^2$, so
$0 \le y^\top P y \le \lVert y\rVert^2$.

*Step 5 — the second inequality.* Using $P^2 = P$ and then Steps 1 and 4,

$$
\operatorname{tr}(PAP) = \operatorname{tr}(P^2A) = \operatorname{tr}(PA) = \operatorname{tr}(L^\top P L)
= \sum_{i=1}^{n} (Le_i)^\top P (Le_i) \ \le \ \sum_{i=1}^{n}\lVert Le_i\rVert^2 = \operatorname{tr}(A) .
$$

$$
\boxed{\operatorname{tr}(AB) \gt 0, \qquad \operatorname{tr}(PAP) \le \operatorname{tr}(A)}
$$

**Key takeaway.** Both proofs avoid eigenvalues entirely: the Cholesky factor from
[Module 03](../03_linear_systems_and_direct_factorizations/first_principles.ipynb) does all the
work, and the projector bound is Theorem 4.4 applied one column at a time.

In [44]:
def rand_spd(n, rng):
    M = rng.standard_normal((n, n))
    return M @ M.T + n * np.eye(n)


A_pd, B_pd = rand_spd(5, rng), rand_spd(5, rng)
L = np.linalg.cholesky(A_pd)
Bp, _ = np.linalg.qr(rng.standard_normal((5, 2)))
P_pr = Bp @ Bp.T
print("||A - L L^T||             :", np.linalg.norm(A_pd - L @ L.T))
print("tr(AB)                    :", np.trace(A_pd @ B_pd))
print("sum (L e_i)^T B (L e_i)   :", sum(L[:, i] @ B_pd @ L[:, i] for i in range(5)))
print("tr(PAP), tr(A)            :", np.trace(P_pr @ A_pd @ P_pr), np.trace(A_pd))
print("tr(PAP) = tr(PA)?         :", np.trace(P_pr @ A_pd @ P_pr), np.trace(P_pr @ A_pd))
assert np.trace(A_pd @ B_pd) > 0
assert abs(np.trace(A_pd @ B_pd) - sum(L[:, i] @ B_pd @ L[:, i] for i in range(5))) < 1e-9
assert np.trace(P_pr @ A_pd @ P_pr) <= np.trace(A_pd) + 1e-10

||A - L L^T||             : 2.012231377123348e-15
tr(AB)                    : 765.466098550494
sum (L e_i)^T B (L e_i)   : 765.4660985504942
tr(PAP), tr(A)            : 26.441099406908137 63.81317901593572
tr(PAP) = tr(PA)?         : 26.441099406908137 26.441099406908137
